# CVD mortality prediction: revision pipeline (JGMS-2026-RES-1180, stepwise version)

The main workflow is not wrapped in functions. Every cell leaves its results in the notebook namespace,
so any cell can be inspected or re-run on its own after an error.

**Tag on the first line of each code cell**

Each code cell starts with `[Cell N]` (code cells numbered 1, 2, 3, ... from the top; markdown cells are not counted), followed by one of:

- `[Original, unchanged]`: copied from `new code.ipynb` (original cell number given).
- `[Original, modified]`: based on the original code; the second line states what changed.
- `[New]`: added for the revision; the second line names the reviewer comment it answers
  (R1.x = Reviewer 1, R2.x = Reviewer 2, Minor = smaller revisions) or marks it as an extra analysis.

**Suggested test order**

1. In section 1 set `QUICK_TEST = True` and `COHORTS_TO_RUN = ["bothsex"]`; run all cells.
2. Set `COHORTS_TO_RUN` back to all three cohorts, keep `QUICK_TEST = True`; run all cells.
3. Set `QUICK_TEST = False` for the final run. Optional analyses can be switched off in `RUN`.

Results are stored in dictionaries keyed by cohort, e.g. `TEST["male"]`, `PRED[("male", "LightGBM", 10.0)]`.

Export only `OUT_DIR/tables/` and `OUT_DIR/figures/`. `OUT_DIR/participant_level/` holds row-level data
and must not leave SURE.

## Section 0. Imports

In [ ]:
# [Cell 1] [Original, modified] new code.ipynb cell 0
# Change: duplicate imports removed; imports used by the revision added; sksurv version printed.
import os, sys, io, json, time, struct, warnings, contextlib
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.special import expit, logit

import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.model_selection import (RandomizedSearchCV, GridSearchCV, StratifiedKFold,
                                     RepeatedStratifiedKFold, train_test_split)
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import lifelines
from lifelines import CoxPHFitter, WeibullAFTFitter
import sksurv
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis
from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
from sksurv.svm import FastSurvivalSVM
from sksurv.metrics import (concordance_index_censored, concordance_index_ipcw,
                            cumulative_dynamic_auc, brier_score)
from sksurv.util import Surv

import lightgbm as lgb
import xgboost as xgb
import shap
import matplotlib
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:          # running as a plain script
    display = print

warnings.filterwarnings("ignore")
VERSIONS_USED = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                 "scipy": scipy.__version__, "sklearn": sklearn.__version__, "sksurv": sksurv.__version__,
                 "lightgbm": lgb.__version__, "xgboost": xgb.__version__, "lifelines": lifelines.__version__,
                 "shap": shap.__version__, "matplotlib": matplotlib.__version__}
print(VERSIONS_USED)
T_START = time.time()

## Section 1. Settings

In [ ]:
# [Cell 2] [Original, modified] new code.ipynb cell 1
# Change: the male file path is removed (the male cohort is rebuilt from the both-sex file, section 4);
#         output folder is outputs_revision_R1; RANGES entries for modtimesperweek, modtotalhrsweek and
#         numredmeattwo removed (not in the data, see 'Valid Data' in the variable list).
COHORT_FILES = {
    "bothsex": Path(r"G:\User space\RongC\Final\RawData\rawData_allsex_98_edate.csv"),
    "female": Path(r"G:\User space\RongC\Final\RawData\rawData_female_100_edate.csv"),
}
BINARY_TXT = Path(r"G:\User space\RongC\Final\RawData\binary-col.txt")
MULTI_TXT = Path(r"G:\User space\RongC\Final\RawData\multi-cat-col.txt")
RAW_FILE = Path(r"G:\User space\RongC\Raw Data\rawData.csv")   # only used by section 23 (optional data checks)
OUT_DIR = Path(r"G:\User space\RongC\Final\outputs_revision_R1")

TIME_COL = "edate"
EVENT_COL = "survival"
AGE_COL = "age_calc"
INVERT_EVENT = False
HORIZONS = [5.0, 10.0]
SEED = 42
N_BOOT = 1000
TEST_SIZE = 0.20
N_BINS = 10
SPECIAL_EXCLUDE = ("survival", "edate")
USE_DTYPE_FOR_NUMERIC = False
SURVIVAL_STRATIFY = True

RANGES = {
    "age_calc": (0, 140),
    "alcdrinksperweek": (0, 140),
    "bmi": (9, 50),
    "brokeboneage": (0, 140),
    "dayhoursscreenwatch": (0, 24),
    "dayhourssitting": (0, 24),
    "dayhourssleeping": (0, 24),
    "dayhoursstanding": (0, 18),
    "fruit": (0, 16),
    "fruitjuice": (0, 16),
    "numbrec": (0, 45),
    "numbrownbread": (0, 140),
    "numcheese": (0, 50),
    "numfish": (0, 50),
    "freq_poultry_week": (0, 50),
    "freq_redmeat_week": (0, 50),
    "numpromeat": (0, 50),
    "vegcooked": (0, 16),
    "sf36score": (0, 100),
    "k10_score": (0, 100),
    "dssi": (0, 30),
    "mammogramtimesnum": (0, 50),
    "vigour": (0, 2000),
    "walk": (0, 2000),
    "moderate": (0, 2000),
    "veg": (0, 140),
    "protein": (0, 140),
    "fruitmet": (0, 140),
}

In [ ]:
# [Cell 3] [New] Revision settings and switches
# Purpose: R1.1 (male cohort rebuilt and checked), test switches, optional analyses.
SEX_COL = "sex"
MALE_CODE = 1                     # value of SEX_COL for men in the both-sex file
PRIMARY = 10.0
CAL_SIZE = 0.20                   # test_size of the second split in the original code
EXPECTED = {"bothsex": (18603, 7308), "female": (6931, 2552), "male": (11672, 4756)}
SUBMITTED_CC_AUROC = {("bothsex", 10.0): 0.846, ("female", 10.0): 0.848,
                      ("bothsex", 5.0): 0.844, ("female", 5.0): 0.838}

COHORTS_TO_RUN = ["bothsex", "female", "male"]   # for a first test use ["bothsex"]
QUICK_TEST = False                               # True: few resamples, runs in minutes
CHECK_EXPECTED_COUNTS = True                     # stop if cohort sizes differ from EXPECTED (full run only)

# Optional analyses (core sections 1-11 always run). Switch off to save time.
RUN = {
    "extra_cox_benchmarks": True,    # section 6b: ridge, lasso and tuned elastic-net Cox (R2.2)
    "spline_benchmarks": True,       # section 6c: restricted cubic spline logistic and Cox models (Minor 3)
    "internal_validation": True,     # section 12: bootstrap optimism + repeated CV (R1.5, R2.2)
    "nested_selection_cv": True,     # section 13: nested CV that repeats algorithm selection (R1.5, R2.2)
    "domain_models": True,           # section 14: demographic -> clinical -> domain-by-domain models (R1.6, R2.7)
    "sex_comparison": True,          # section 15: pooled vs interaction vs sex-specific on identical people (R1.7, R2.8)
    "missing_data": True,            # section 16: multiple imputation, native missing, missing indicators (R1.8, R2.5)
    "weighting_sensitivity": True,   # section 17: censoring / class-weight training variants (R1.2, R2.1)
    "tuning_sensitivity": True,      # section 18: cross-validated tuning vs fixed settings (R2.2)
    "repeated_splits": True,         # section 19: model comparison over repeated random splits (R1.4)
    "shap_stability": True,          # section 20: bootstrap SHAP ranks, sex rank differences, permutation importance (R2.8)
    "subgroups": True,               # section 21: age-group / sex subgroups, age >= 65 refit (Minor 1)
    "cox_hr": True,                  # section 22: ridge Cox hazard ratios (eFigure 6) and sex-interaction tests (R1.7)
    "data_checks": True,             # section 23: CVD variables and SF-36 definition (R2.3, R2.4)
}

N_BOOT_OPT = 200                  # bootstrap resamples for optimism correction
N_BOOT_SECONDARY = 200            # bootstrap resamples for large secondary tables
CV_FOLDS, CV_REPEATS = 5, 5       # repeated CV for internal validation
NESTED_FOLDS, NESTED_REPEATS = 5, 2
M_IMPUTATIONS = 10
N_SPLIT_SEEDS = 10                # repeated random train/test splits
N_SHAP_BOOT = 20                  # bootstrap refits for SHAP rank stability
PERM_REPEATS = 5                  # permutation importance repeats
TUNE_N_ITER, TUNE_FOLDS = 30, 5   # randomized search for the tuning sensitivity analysis
OPTIMISM_COHORTS = list(COHORTS_TO_RUN)
NESTED_COHORTS = list(COHORTS_TO_RUN)
SPLIT_COHORTS = list(COHORTS_TO_RUN)
REPEATED_SPLIT_MODELS = ["LightGBM", "XGBoost", "RF", "LogReg", "CoxNet"]
if QUICK_TEST:
    N_BOOT, N_BOOT_OPT, N_BOOT_SECONDARY = 50, 5, 30
    CV_REPEATS, NESTED_REPEATS, M_IMPUTATIONS = 1, 1, 2
    N_SPLIT_SEEDS, N_SHAP_BOOT, PERM_REPEATS, TUNE_N_ITER, TUNE_FOLDS = 2, 3, 2, 3, 3

SCHEMES = ["ipcw", "cc"]          # ipcw: IPCW-weighted, all test participants; cc: complete case (known status only)
PRIMARY_SCHEME = "ipcw"           # scheme used for recalibration in derived analyses and for the main figures
DCA_THRESHOLDS = np.round(np.arange(0.01, 0.81, 0.01), 2)
OP_THRESHOLDS = {"7.5%": 0.075, "20%": 0.20}
NB_THRESHOLDS = [0.10, 0.20, 0.30]
MULTI_NA_CODES = (99999, "99999")
SMALL_CELL = 5                    # counts 1-4 are shown as "<5" in exported tables
AGE_BANDS = [0, 55, 65, 75, 85, np.inf]
AGE_BAND_LABELS = ["45-54", "55-64", "65-74", "75-84", ">=85"]
SUBGROUP_AGE = [0, 65, 75, np.inf]
SUBGROUP_AGE_LABELS = ["<65", "65-74", ">=75"]

In [ ]:
# [Cell 4] [New] Variable sets, domains and display names
# Purpose: R1.6 (domain models), R1.8 (predictors tracked in missing-data analyses), Figure 3 labels.
DEMOGRAPHIC_VARS = ["age_calc", "sex"]
CLINICAL_VARS = ["diabetes", "stroke", "AF", "ht", "bp", "treatbp", "treathd", "treathattack",
                 "treatbchol", "aspirinhrt", "smoke", "smoke2cat", "bmi", "alcdrinksperweek",
                 "heartdis", "diagnosisP"]
# Domains added one at a time after the clinical model (order requested by R1.6). Any predictor not listed
# here is added in the last step ("remaining") and is printed in section 4 so the grouping can be checked.
DOMAIN_VARS = {
    "functional": ["sf36score", "limitvigor", "limitmod", "limitlift", "limitupfew", "limitupone",
                   "limitbend", "limit1km", "limithalfkm", "limit100m", "limitdress"],
    "perceived health": ["ratehealth", "ratequality", "ratememory"],
    "sensory": ["hearinglossyn"],
    "oral": ["teeth", "rateteeth"],
    "social": ["marital", "dssi"],
    "dietary": ["cerebiscuit", "cerebran", "ceremuesli", "cereother", "cereoat", "milksoy", "milkskim",
                "milkother", "milkreducedfat", "milkwhole", "no_milk", "never_any_meat", "nosugar", "nodairy",
                "noseafood", "nowheat", "noeggs", "nocream", "fruit", "fruitjuice", "fruitmet", "numbrec",
                "numbrownbread", "numcheese", "numfish", "freq_poultry_week", "freq_redmeat_week", "numpromeat",
                "vegcooked", "vegraw", "veg", "protein"],
}
DOMAIN_ORDER = ["functional", "perceived health", "sensory", "oral", "social", "dietary"]
KEY_PREDICTORS = ["age_calc", "sf36score", "workstat_1.0", "diabetes", "hearinglossyn",
                  "teeth_4.0", "aspirinhrt", "mammogramtimesnum", "fruitjuice", "numbrec",
                  "numpromeat", "dayhoursstanding", "patimeh", "bmi"]
LABELS = {
    "age_calc": "Age", "sex": "Sex (male)", "bmi": "BMI",
    "sf36score": "SF-36 physical functioning score",
    "workstat_1.0": "Work status: retired/not working", "workstat_2.0": "Work status: part-time",
    "workstat_3.0": "Work status: self-employed/other",
    "ratehealth_1.0": "Self-rated health: excellent", "ratehealth_2.0": "Self-rated health: very good",
    "ratehealth_3.0": "Self-rated health: good", "ratehealth_4.0": "Self-rated health: fair",
    "ratehealth_5.0": "Self-rated health: poor",
    "teeth_1.0": "Teeth: none", "teeth_2.0": "Teeth: 1-9", "teeth_3.0": "Teeth: 10-19",
    "teeth_4.0": "Teeth: 20 or more",
    "hearinglossyn": "Self-reported hearing loss", "diabetes": "Diagnosed diabetes",
    "aspirinhrt": "Aspirin use for CVD", "fruitjuice": "Fruit juice (servings/day)",
    "Index_RSE_Disadvantage": "Socioeconomic disadvantage index",
    "dayhourssleeping": "Sleep (hours/day)", "dayhourssitting": "Sitting (hours/day)",
    "dayhoursstanding": "Standing (hours/day)", "milkwhole": "Whole milk",
    "patimeh": "Weighted physical activity (hours/week)",
    "mammogramtimesnum": "Lifetime number of mammograms",
    "numbrec": "Breakfast cereal (times/week)", "numpromeat": "Processed meat (times/week)",
    "nosugar": "No added sugar", "vigour": "Vigorous activity (minutes/week)",
    "treatbchol": "Treated for high cholesterol (last month)", "vegcooked": "Cooked vegetables (servings/day)",
    "limit100m_3.0": "Not limited walking 100 m", "limit1km_1.0": "Limited a lot walking 1 km",
    "limithalfkm_1.0": "Limited a lot walking 500 m", "marital_1.0": "Married/partner",
    "marital_3.0": "Never married", "k10_score": "Psychological distress (K10)",
    "alcdrinksperweek": "Alcohol (drinks/week)", "brokeboneage": "Age at broken bone",
    "dayhoursscreenwatch": "Screen time (hours/day)", "dssi": "Duke Social Support Index",
    "freq_poultry_week": "Poultry (times/week)", "freq_redmeat_week": "Red meat (times/week)",
    "fruit": "Fruit (servings/day)", "fruitmet": "Meets fruit recommendation",
    "moderate": "Moderate activity (minutes/week)", "walk": "Walking (minutes/week)",
    "numbrownbread": "Brown/wholemeal bread (servings/week)", "numcheese": "Cheese (times/week)",
    "numfish": "Fish or seafood (times/week)", "protein": "Protein intake indicator",
    "veg": "Vegetable intake indicator", "vegraw": "Raw vegetables (servings/day)",
    "stroke": "Diagnosed stroke", "AF": "Atrial fibrillation", "smoke": "Current smoking",
    "treatbp": "Treated for high blood pressure (last month)", "ht": "Hypertension",
    # [R1 label fix] labels missing from Figure 3C and eFigure 6 (display only; no effect on any result)
    "limitupone_3.0": "Not limited climbing 1 flight of stairs", "limitmod_1.0": "Limited a lot moderate activities",
    "limitvigor_3.0": "Not limited vigorous activities", "limit100m_1.0": "Limited a lot walking 100 m",
    "mammogramyn": "Ever had screening mammogram", "breastcancer": "Diagnosed breast cancer",
    "suff_pa": "Sufficient physical activity", "smoke2cat": "Smoking status",
    "fhAlzhbrosis": "Family history of dementia: sibling", "milksoy": "Soy milk",
    "treathd": "Treated for other heart disease (last month)",
    "highestqual_4.0": "Highest qualification: trade/apprenticeship",
}
MODEL_LABEL = {"CoxNet": "Penalized Cox (elastic net)", "WeibullAFT": "Weibull AFT",
               "RSF": "Random survival forest", "GBSurv": "Gradient-boosted survival",
               "FastSurvSVM": "Survival SVM (fast)", "XGBoost-Cox": "XGBoost survival (Cox)",
               "LogReg": "Logistic regression", "GNB": "Gaussian naive Bayes",
               "MLP": "Multilayer perceptron", "RF": "Random forest", "LightGBM": "LightGBM",
               "XGBoost": "XGBoost (classifier)",
               "CoxNet-EN-tuned": "Elastic-net Cox (penalty tuned)", "CoxNet-lasso": "Lasso Cox (penalty tuned)",
               "Cox-ridge": "Ridge Cox (penalty tuned)", "LogReg-spline": "Logistic regression (splines)",
               "CoxNet-spline": "Elastic-net Cox (splines)", "LightGBM-10y-model": "LightGBM (10-year model)"}
COHORT_LABEL = {"bothsex": "Both sexes", "female": "Females", "male": "Males"}
COHORT_COLOR = {"bothsex": "#000000", "female": "#D55E00", "male": "#0072B2"}

In [ ]:
# [Cell 5] [Original, modified] new code.ipynb cell 1 (folder creation)
# Change: subfolders are tables / figures / participant_level.
TABLES, FIGS, ROWLEVEL = OUT_DIR / "tables", OUT_DIR / "figures", OUT_DIR / "participant_level"
for d in (TABLES, FIGS, ROWLEVEL):
    d.mkdir(parents=True, exist_ok=True)
print("output dir:", OUT_DIR)
for k, v in COHORT_FILES.items():
    print(f" {k:<8} exists: {v.exists()} -> {v}")
print("cohorts to run:", COHORTS_TO_RUN, " QUICK_TEST:", QUICK_TEST, " N_BOOT:", N_BOOT)
print("optional analyses on:", [k for k, v in RUN.items() if v])
TIMINGS = {}

## Section 2. Original helper functions (copied unchanged)

In [ ]:
# [Cell 6] [Original, unchanged] new code.ipynb cell 2: variable groups and preprocessing
@dataclass
class PreprocessArtifacts:
    binary_fill_: Dict[str, float] = field(default_factory=dict)
    num_imputer_: Optional[SimpleImputer] = None
    cat_imputer_: Optional[SimpleImputer] = None
    scaler_: Optional[StandardScaler] = None
    ohe_: Optional[OneHotEncoder] = None
    ohe_feature_names_: Optional[np.ndarray] = None
    config_: Dict = field(default_factory=dict)
    range_clean_log_: Dict[str, pd.DataFrame] = field(default_factory=dict)

def load_list(path):
    names = []
    with open(path, 'r', encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            names.append(s.strip().strip("'"))
    return names

def intersect_present(candidates, present):
    return sorted(set(candidates) & present)

def build_column_buckets(df, binary_txt_path, multi_txt_path, special_exclude=SPECIAL_EXCLUDE, use_dtype_for_numeric=False):
    df_cols = set(df.columns)
    bin_set = load_list(binary_txt_path)
    multi_set = load_list(multi_txt_path)
    binary_cols = intersect_present(bin_set, df_cols)
    multi_cat_cols = intersect_present(multi_set, df_cols)
    special_cols = intersect_present(special_exclude, df_cols)

    if use_dtype_for_numeric:
        num_all = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
        num_cols = sorted(set(num_all) - set(binary_cols) - set(multi_cat_cols) - set(special_cols))
    else:
        covered = set(binary_cols) | set(multi_cat_cols) | set(special_cols)
        num_cols = sorted(df_cols - covered)
    missing_bucket = sorted(df_cols - (set(num_cols) | set(binary_cols) | set(multi_cat_cols) | set(special_cols)))

    if missing_bucket:
        print("WARNING: Columns not assigned to any bucket", missing_bucket)
    print(f"[SUMMARY] binary={len(binary_cols)}, multi={len(multi_cat_cols)},"
          f"numeric={len(num_cols)}, special={len(special_cols)}")
    return binary_cols, num_cols, multi_cat_cols, special_cols

def preprocess_for_model(
    train_df, test_df, *, binary_cols, num_cols, multi_cat_cols,
    passthrough_cols=None, scale_numeric=True,
    cat_special_na_values=(99999, "99999"), ranges=None):

    artifacts = PreprocessArtifacts()
    artifacts.config_ = {
        "scale_numeric": scale_numeric,
        "cat_special_na_values": cat_special_na_values,
        "binary_cols": list(binary_cols),
        "num_cols": list(num_cols),
        "multi_cat_cols": list(multi_cat_cols)}

    train = train_df.copy(); test = test_df.copy()
    PT = set(passthrough_cols or [])
    bin_cols = [c for c in binary_cols if c in train.columns and c not in PT]
    num_cols = [c for c in num_cols if c in train.columns and c not in PT]
    cat_cols = [c for c in multi_cat_cols if c in train.columns and c not in PT]

    for col in bin_cols:
        modes = train[col].dropna().mode()
        fill_val = modes.iloc[0] if len(modes) else 0
        artifacts.binary_fill_[col] = fill_val
        train[col] = train[col].where(train[col].notna(), fill_val)
        test[col] = test[col].where(test[col].notna(), fill_val)

    def _oor(df_, ranges, cols):
        df_ = df_.copy(); log = []
        for col in cols:
            if col not in df_.columns or col not in ranges: continue
            s = df_[col]
            if not pd.api.types.is_numeric_dtype(s):
                log.append({"col": col, "n_set_to_nan": 0, "note": "non-numeric"}); continue
            lo, hi = ranges[col]
            if lo is None or hi is None or not (lo < hi):
                log.append({"col": col, "n_set_to_nan": 0, "note": "non-numeric"}); continue
            mask = s.notna() & ((s < lo) | (s > hi))
            n = int(mask.sum())
            if n: df_.loc[mask, col] = np.nan
            log.append({"col": col, "lo": lo, "hi": hi, "n_set_to_nan": n})
        return df_, pd.DataFrame(log).sort_values("n_set_to_nan", ascending=False)

    if ranges and num_cols:
        cols_to_check = [c for c in num_cols if c in ranges]
        train, lt = _oor(train, ranges, cols_to_check)
        test, le = _oor(test, ranges, cols_to_check)
        artifacts.range_clean_log_ = {"train": lt, "test": le}

    if num_cols:
        ni = SimpleImputer(strategy='median')
        train.loc[:, num_cols] = ni.fit_transform(train[num_cols])
        test.loc[:, num_cols] = ni.transform(test[num_cols])
        artifacts.num_imputer_ = ni

    if cat_cols:
        if cat_special_na_values:
            repl = {v: np.nan for v in cat_special_na_values}
            for d in (train, test):
                d.loc[:, cat_cols] = d[cat_cols].replace(repl)
        ci = SimpleImputer(strategy='most_frequent')
        train.loc[:, cat_cols] = ci.fit_transform(train[cat_cols])
        test.loc[:, cat_cols] = ci.transform(test[cat_cols])
        artifacts.cat_imputer_ = ci

    if scale_numeric and num_cols:
        sc = StandardScaler()
        train.loc[:, num_cols] = sc.fit_transform(train[num_cols])
        test.loc[:, num_cols] = sc.transform(test[num_cols])
        artifacts.scaler_ = sc

    if cat_cols:
        try:
            ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        except TypeError:
            ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
        train_ohe = pd.DataFrame(
            ohe.fit_transform(train[cat_cols]),
            columns=ohe.get_feature_names_out(cat_cols),
            index=train.index)
        test_ohe = pd.DataFrame(
            ohe.transform(test[cat_cols]),
            columns=ohe.get_feature_names_out(cat_cols),
            index=test.index)
        artifacts.ohe_ = ohe
        artifacts.ohe_feature_names_ = ohe.get_feature_names_out(cat_cols)
        train = train.drop(columns=cat_cols).join(train_ohe)
        test = test.drop(columns=cat_cols).join(test_ohe)

    train_aligned, test_aligned = train.align(test, join="left", axis=1, fill_value=0)
    return train_aligned, test_aligned, artifacts
print("preprocess_for_model + build_column_buckets OK")

In [ ]:
# [Cell 7] [Original, modified] new code.ipynb cell 3: X/y builders, fixed-horizon labels, C-index
# Change: shorter warning text; cindex_auto no longer returns the unused uno_se; unused helpers removed.
def build_Xy_survival_sksurv(train_df_proc, test_df_proc, time_col, event_col, feature_exclude=None):
    exclude = set([time_col, event_col] + list(feature_exclude or []))

    y_train = Surv.from_arrays(
        event=train_df_proc[event_col].astype(bool).values,
        time=train_df_proc[time_col].astype(float).values,
    )
    y_test = Surv.from_arrays(
        event=test_df_proc[event_col].astype(bool).values,
        time=test_df_proc[time_col].astype(float).values,
    )
    X_train = train_df_proc.drop(columns=list(exclude))
    X_test = test_df_proc.drop(columns=list(exclude))
    return X_train, X_test, y_train, y_test

def make_horizon_dataset(frame, t_years, time_col="edate", event_col="survival", feature_exclude=None, label_frame=None):
    """Drop pre-horizon censorings (NaN labels)."""
    feature_exclude = list(feature_exclude or [])
    src = label_frame if label_frame is not None else frame
    if time_col not in src.columns:
        raise KeyError(f"make_horizon_dataset: '{time_col}' not in label-frame columns")
    if event_col not in src.columns:
        raise KeyError(f"make_horizon_dataset: '{event_col}' not in label-frame columns")
    time = src[time_col].astype(float).values
    event = src[event_col].astype(float).values
    died_by = (event == 1) & (time <= t_years)
    alive_at = (time > t_years)
    keep = died_by | alive_at
    y_t = np.where(died_by[keep], 1, 0).astype(int)
    if len(np.unique(y_t)) < 2:
        print(f"[WARN make_horizon_dataset h={t_years}] degenerate y: "
              f"died_by={int(died_by.sum())}, alive_at={int(alive_at.sum())}, keep={int(keep.sum())}")
    drop_cols = [c for c in [time_col, event_col] + feature_exclude if c in frame.columns]
    X_t = frame.loc[keep].drop(columns=drop_cols)
    return X_t, y_t

def cindex_auto(y_train_surv, y_test_surv, risk):
    """Harrell's + Uno's IPCW-weighted C-index; auto-flip sign if needed."""
    risk = np.asarray(risk).ravel().astype(float)
    evt = y_test_surv["event"].astype(bool)
    tim = y_test_surv["time"].astype(float)
    c1 = concordance_index_censored(evt, tim, risk)[0]
    c2 = concordance_index_censored(evt, tim, -risk)[0]
    flipped = c2 > c1
    risk_used = -risk if flipped else risk
    harrell = float(c2 if flipped else c1)
    ipcw = concordance_index_ipcw(y_train_surv, y_test_surv, risk_used)
    return {"harrell_c": harrell, "uno_c": float(ipcw[0]), "flipped": bool(flipped)}
print("build_Xy + horizon helpers OK")

In [ ]:
# [Cell 8] [Original, modified] new code.ipynb cell 6 (first part): classification models
# Change: settings unchanged; model lists and new_clf() added (XGBoost positive-class weight from the training labels, as in eval_classifier_horizon).
def make_clf_models(seed, n_neg, n_pos):
    spw = float(n_neg) / max(float(n_pos), 1)
    return {
        "LogReg": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=seed, solver="liblinear"),
        "GNB": GaussianNB(),
        "MLP": MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu",
                             alpha=1e-4, early_stopping=True, max_iter=300, random_state=seed),
        "RF": RandomForestClassifier(n_estimators=500, max_depth=None, n_jobs=-1,
                                     class_weight="balanced_subsample", random_state=seed),
        "LightGBM": lgb.LGBMClassifier(
            random_state=seed,
            n_estimators=600,
            learning_rate=0.03,
            num_leaves=31,
            subsample=0.8,
            max_depth=3,
            colsample_bytree=0.8,
            is_unbalance=True,
            verbosity=-1
        ),
        "XGBoost": xgb.XGBClassifier(random_state=seed,
                                     n_estimators=400,
                                     learning_rate=0.05,
                                     tree_method="hist",
                                     eval_metric="auc",
                                     max_depth=5,
                                     scale_pos_weight=spw
                                     ),
    }
TREE_MODELS = ("RF", "LightGBM", "XGBoost")
CLF_ORDER = ["LogReg", "GNB", "MLP", "RF", "LightGBM", "XGBoost"]
SURV_NAMES = ["CoxNet", "WeibullAFT", "RSF", "GBSurv", "FastSurvSVM", "XGBoost-Cox"]


def new_clf(name, y_train):
    """Fresh classifier with the original settings (XGBoost positive-class weight from y_train)."""
    y_train = np.asarray(y_train)
    return make_clf_models(SEED, int((y_train == 0).sum()), int((y_train == 1).sum()))[name]

## Section 3. New helper functions (censoring, weighted metrics, recalibration, bootstrap, output)

Two evaluation schemes are produced for every fixed-horizon metric:

- `ipcw`: all test participants; death by the horizon has weight 1/G(T-), alive beyond the horizon has
  weight 1/G(h), censored before the horizon has weight 0. G is the Kaplan-Meier estimate of the
  censoring distribution in the training set.
- `cc` (complete case): participants censored before the horizon are excluded, all others have weight 1.
  This is how the submitted manuscript evaluated the classifiers.

In [ ]:
# [Cell 9] [New] Censoring distribution and fixed-horizon labels / weights
# Purpose: R1.2, R2.1 (how the fixed-horizon outcome and censoring were handled).
class CensoringKM:
    """Reverse Kaplan-Meier G(t) = P(C > t) from training data. Deaths tied with censorings are treated
    as occurring first (same convention as scikit-survival)."""
    def __init__(self, time, event):
        time = np.asarray(time, float)
        event = np.asarray(event).astype(int)
        uniq = np.unique(time)
        n_ge = len(time) - np.searchsorted(np.sort(time), uniq, side="left")
        deaths = pd.Series(event == 1).groupby(time).sum().reindex(uniq, fill_value=0).values
        cens = pd.Series(event == 0).groupby(time).sum().reindex(uniq, fill_value=0).values
        at_risk = n_ge - deaths
        with np.errstate(divide="ignore", invalid="ignore"):
            step = np.where(at_risk > 0, 1 - cens / at_risk, 1.0)
        self.times, self.surv = uniq, np.cumprod(step)

    def G(self, t, left=False):
        t = np.asarray(t, float)
        idx = np.searchsorted(self.times, t, side="left" if left else "right") - 1
        return np.clip(np.where(idx >= 0, self.surv[np.clip(idx, 0, None)], 1.0), 1e-6, None)


def horizon_outcome(time, event, h, km):
    """Per-person fixed-horizon label y, IPCW weight w and known-status flag.
    Died by h: y=1, w=1/G(T-). Followed beyond h: y=0, w=1/G(h). Censored before h: w=0, known=False."""
    time, event = np.asarray(time, float), np.asarray(event).astype(int)
    died, alive = (event == 1) & (time <= h), time > h
    w = np.zeros(len(time))
    w[died] = 1.0 / km.G(time[died], left=True)
    w[alive] = 1.0 / km.G(np.full(alive.sum(), h))
    return died.astype(int), w, (died | alive)


def scheme_weights(w_ipcw, known, scheme):
    return w_ipcw if scheme == "ipcw" else known.astype(float)


def horizon_strata(y, known):
    """0 = censored before the horizon, 1 = alive at the horizon, 2 = died by the horizon."""
    return np.where(~known, 0, np.where(y == 1, 2, 1))

In [ ]:
# [Cell 10] [New] Weighted discrimination, Brier score and calibration measures
# Purpose: R1.2, R1.3, R2.6. With complete-case weights (0/1) they equal the ordinary unweighted measures.
def w_auc(y, s, w):
    m = w > 0
    return roc_auc_score(y[m], s[m], sample_weight=w[m])

def w_ap(y, p, w):
    m = w > 0
    return average_precision_score(y[m], p[m], sample_weight=w[m])

def w_brier(y, p, w):
    """IPCW Brier score (Graf): sum w (y - p)^2 / n over all participants. With 0/1 weights the
    denominator is the number with known status, i.e. the ordinary Brier score."""
    w = np.asarray(w, float)
    denom = np.sum(w > 0) if np.all((w == 0) | (w == 1)) else len(y)
    return float(np.sum(w * (y - p) ** 2) / denom)

def w_prevalence(y, w):
    return float(np.sum(w * y) / np.sum(w))

def w_scaled_brier(y, p, w):
    """Index of prediction accuracy: 1 - Brier / Brier of the null model (constant observed risk)."""
    b0 = w_brier(y, np.full(len(y), w_prevalence(y, w)), w)
    return float(1 - w_brier(y, p, w) / b0) if b0 > 0 else np.nan

def _wlogit_fit(X, y, w, offset=None, iters=100):
    X = np.column_stack([np.ones(len(y))] + ([X] if X is not None else []))
    off = np.zeros(len(y)) if offset is None else offset
    beta = np.zeros(X.shape[1])
    for _ in range(iters):
        p = expit(np.clip(X @ beta + off, -30, 30))
        step = np.linalg.solve(X.T @ (X * (w * p * (1 - p))[:, None]) + 1e-10 * np.eye(len(beta)),
                               X.T @ (w * (y - p)))
        beta += step
        if np.max(np.abs(step)) < 1e-8:
            break
    return beta

def lp(p):
    return logit(np.clip(np.asarray(p, float), 1e-7, 1 - 1e-7))

def cal_intercept_w(y, p, w):
    """Calibration intercept (calibration-in-the-large): slope fixed at 1, logit(p) as offset."""
    m = w > 0
    return float(_wlogit_fit(None, y[m], w[m], offset=lp(p[m]))[0])

def cal_slope_w(y, p, w):
    """Calibration slope: logistic regression of the outcome on logit(p) (intercept estimated jointly)."""
    m = w > 0
    return float(_wlogit_fit(lp(p[m]), y[m], w[m])[1])

def oe_ratio(y, p, w):
    """Observed / expected (weighted observed risk / weighted mean predicted risk)."""
    m = w > 0
    return float(w_prevalence(y[m], w[m]) / np.average(p[m], weights=w[m]))

def w_ece(y, p, w, n_bins=N_BINS):
    """Weighted expected calibration error over deciles of predicted risk."""
    rt, _ = reliability_table_w(y, p, w, n_bins)
    return float(np.average(np.abs(rt["obs"] - rt["pred_mean"]), weights=rt["w_sum"]))

def reliability_table_w(y, p, w, n_bins=N_BINS, edges=None):
    m = w > 0
    if edges is None:
        edges = np.unique(np.quantile(p[m], np.linspace(0, 1, n_bins + 1)))
    b = np.clip(np.digitize(p, edges[1:-1], right=True), 0, len(edges) - 2)
    rows = []
    for k in range(len(edges) - 1):
        mk = m & (b == k)
        if mk.sum():
            rows.append({"bin": k + 1, "n": int(mk.sum()), "w_sum": float(w[mk].sum()),
                         "pred_mean": float(np.average(p[mk], weights=w[mk])),
                         "obs": float(np.average(y[mk], weights=w[mk]))})
    return pd.DataFrame(rows), edges

METRICS = {"AUROC": w_auc, "PR_AUC": w_ap, "Brier": w_brier, "scaled_Brier": w_scaled_brier,
           "cal_intercept": cal_intercept_w, "cal_slope": cal_slope_w, "OE_ratio": oe_ratio, "ECE": w_ece}

In [ ]:
# [Cell 11] [New] Weighted recalibration, decision curves and operating characteristics
# Purpose: R1.3, R2.6 (before / after recalibration), R1.2 (censoring in DCA and operating characteristics).
class Platt:
    def fit(self, p, y, w):
        m = w > 0
        self.a, self.b = _wlogit_fit(lp(p[m]), y[m], w[m])
        return self
    def __call__(self, p):
        return expit(self.a + self.b * lp(p))

class Isotonic:
    def fit(self, p, y, w):
        m = w > 0
        self.iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p[m], y[m], sample_weight=w[m])
        return self
    def __call__(self, p):
        return np.clip(self.iso.predict(p), 1e-6, 1 - 1e-6)

def net_benefit(y, p, w, t):
    W = np.sum(w)
    tp = np.sum(w * ((p >= t) & (y == 1))) / W
    fp = np.sum(w * ((p >= t) & (y == 0))) / W
    return float(tp - fp * t / (1 - t))

def treat_all_nb(y, w, t):
    prev = w_prevalence(y, w)
    return float(prev - (1 - prev) * t / (1 - t))

def youden_threshold_w(y, p, w):
    m = w > 0
    fpr, tpr, thr = roc_curve(y[m], p[m], sample_weight=w[m])
    return float(min(thr[int(np.argmax(tpr - fpr))], 1.0))

def operating_point(y, p, w, t, known):
    pred = p >= t
    tp, fp = np.sum(w * (pred & (y == 1))), np.sum(w * (pred & (y == 0)))
    tn, fn = np.sum(w * (~pred & (y == 0))), np.sum(w * (~pred & (y == 1)))
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    return {"threshold": t, "sensitivity": sens, "specificity": spec,
            "PPV": tp / (tp + fp) if tp + fp else np.nan, "NPV": tn / (tn + fn) if tn + fn else np.nan,
            "F1": 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan,
            "balanced_accuracy": (sens + spec) / 2, "flagged_pct": 100 * np.sum(w * pred) / np.sum(w),
            "TP_n": int(np.sum(known & pred & (y == 1))), "FP_n": int(np.sum(known & pred & (y == 0))),
            "TN_n": int(np.sum(known & ~pred & (y == 0))), "FN_n": int(np.sum(known & ~pred & (y == 1)))}

In [ ]:
# [Cell 12] [New] Bootstrap, formatting, figure saving, small-count suppression
# Purpose: R1.3, R1.4 (CIs and paired differences); journal figure format; SURE output checking.
def boot_idx(strata, B, seed=SEED):
    """Stratified bootstrap indices (resampling within each stratum keeps the numbers of deaths, survivors
    and early censorings fixed). The same indices are used for every model -> paired comparisons."""
    strata = np.asarray(strata)
    rng = np.random.default_rng(seed)
    groups = [np.where(strata == s)[0] for s in np.unique(strata)]
    return np.stack([np.concatenate([rng.choice(g, len(g), replace=True) for g in groups]) for _ in range(B)])

def ci(values):
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) > 10 else (np.nan, np.nan)

def boot_metric(fn, idx, *arrays):
    out = np.empty(len(idx))
    for b, ii in enumerate(idx):
        try:
            out[b] = fn(*[a[ii] for a in arrays])
        except Exception:
            out[b] = np.nan
    return out

def fmt(point, lo_hi, d=3):
    return f"{point:.{d}f} ({lo_hi[0]:.{d}f} to {lo_hi[1]:.{d}f})"

def metric_row(y, p, w, idx, metrics, prefix=""):
    """Point estimate, bootstrap 95% CI and formatted text for each metric."""
    row = {}
    for name in metrics:
        fn = METRICS[name]
        try:
            pt = fn(y, p, w)
        except Exception:
            pt = np.nan
        lo, hi = ci(boot_metric(fn, idx, y, p, w)) if idx is not None else (np.nan, np.nan)
        row.update({f"{prefix}{name}": pt, f"{prefix}{name}_lo": lo, f"{prefix}{name}_hi": hi,
                    f"{prefix}{name}_fmt": fmt(pt, (lo, hi))})
    return row

def save_fig(fig, name, width_in=6.7, max_height_in=8.27, tight=False):
    """Save a 600-dpi PNG with matplotlib only. Convert to a 300-dpi TIFF outside SURE. Figures are drawn at
    the final journal width (170 mm = 6.7 in) so 8-pt fonts stay 8 pt; the printed size is checked."""
    png = FIGS / f"{name}.png"
    fig.savefig(png, dpi=600, bbox_inches="tight" if tight else None, pad_inches=0.03)
    plt.close(fig)
    with open(png, "rb") as fh:
        w_px, h_px = struct.unpack(">II", fh.read(24)[16:24])
    w_in, h_in = w_px / 600, h_px / 600
    note = ""
    if w_in > width_in + 0.05:
        note += f"  <-- wider than {width_in} in: fonts shrink by {width_in / w_in:.2f} when resized"
    if h_in * min(1, width_in / w_in) > max_height_in:
        note += "  <-- taller than 210 mm"
    print(f"  saved {name}.png: {w_in:.2f} x {h_in:.2f} in{note}")

def suppress_small(df, cols=None):
    """Return a copy in which counts 1..SMALL_CELL-1 are replaced by '<SMALL_CELL' (SURE output rule).
    Count columns are detected by name if cols is None."""
    out = df.copy()
    if cols is None:
        keys = ("_n", "n_", "count", "deaths", "TP", "FP", "TN", "FN", "events", "alive", "censored")
        cols = [c for c in out.columns if (c == "n" or any(k in str(c) for k in keys)) and "pct" not in str(c)
                and pd.api.types.is_numeric_dtype(out[c])
                and np.all(np.isclose(out[c].dropna() % 1, 0))]
    for c in cols:
        if c in out.columns and pd.api.types.is_numeric_dtype(out[c]):
            v = out[c]
            out[c] = v.astype(object).where(~((v > 0) & (v < SMALL_CELL)), f"<{SMALL_CELL}")
    return out

def save_table(df, name, suppress=True):
    (suppress_small(df) if suppress else df).to_csv(TABLES / f"{name}.csv", index=False)
    print(f"  saved {name}.csv {df.shape}")

def parent_of(col, multi_cols):
    for m in multi_cols:
        if col.startswith(m + "_"):
            return m
    return col

def shap_matrix(model, X):
    """TreeSHAP values on the log-odds scale (tree-path-dependent, no background data)."""
    sv = shap.TreeExplainer(model).shap_values(X)
    if isinstance(sv, list):
        sv = sv[1]
    elif getattr(sv, "ndim", 2) == 3:
        sv = sv[..., 1]
    return np.asarray(sv)

def shap_direction(x, s):
    """Binary: mean SHAP when 1 minus mean SHAP when 0. Continuous: correlation of value and SHAP.
    Missing feature values (only in the no-imputation analysis) are ignored."""
    x, s = np.asarray(x, float), np.asarray(s, float)
    ok = np.isfinite(x)
    x, s = x[ok], s[ok]
    if set(np.unique(x)) <= {0, 1}:
        on, off = x == 1, x == 0
        return float(s[on].mean() - s[off].mean()) if on.any() and off.any() else 0.0
    return float(np.corrcoef(x, s)[0, 1]) if np.std(x) > 0 and np.std(s) > 0 else 0.0

def direction_label(d):
    return "higher predicted risk" if d > 0 else ("lower predicted risk" if d < 0 else "none")

def tick(label):
    TIMINGS[label] = round((time.time() - T_START) / 60, 1)
    print(f"[{label}] elapsed {TIMINGS[label]} min")

plt.rcParams.update({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "xtick.labelsize": 8,
                     "ytick.labelsize": 8, "legend.fontsize": 8, "font.family": "DejaVu Sans",
                     "figure.dpi": 100})
print("new helpers OK")

In [ ]:
# [Cell 13] [New] Survival-model probabilities at fixed horizons and restricted cubic splines
# Purpose: R2.6 (conversion of survival-model output to fixed-horizon probabilities), Minor 3 (non-linear terms).
def stepfun_at(fns, times):
    """Evaluate scikit-survival StepFunction objects at the given times (right-continuous; 1 before the first
    event time)."""
    times = np.asarray(times, float)
    out = np.empty((len(fns), len(times)))
    for i, f in enumerate(fns):
        idx = np.searchsorted(f.x, times, side="right") - 1
        val = np.where(idx >= 0, f.y[np.clip(idx, 0, None)], 1.0)
        out[i] = f.a * val + f.b
    return out

def surv_prob_sksurv(model, X, times, chunk=500, **kw):
    """P(death by t) = 1 - S(t | x), computed in chunks to limit memory (random survival forest)."""
    parts = []
    for s in range(0, len(X), chunk):
        parts.append(1 - stepfun_at(model.predict_survival_function(X.iloc[s:s + chunk], **kw), times))
    return np.vstack(parts)

def breslow_prob(t_tr, e_tr, hr_tr, hr_new, times):
    """Breslow baseline cumulative hazard from training data; P(death by t | x) = 1 - exp(-H0(t) * HR(x))."""
    t_tr, e_tr, hr_tr = np.asarray(t_tr, float), np.asarray(e_tr).astype(bool), np.asarray(hr_tr, float)
    ev_times = np.unique(t_tr[e_tr])
    d = pd.Series(e_tr.astype(int)).groupby(t_tr).sum().reindex(ev_times).values
    order = np.argsort(t_tr)
    ts, hs = t_tr[order], hr_tr[order]
    tail = np.cumsum(hs[::-1])[::-1]                         # sum of HR over T >= ts[i]
    risk_set = tail[np.searchsorted(ts, ev_times, side="left")]
    H0 = np.cumsum(d / risk_set)
    H_at = np.array([H0[np.searchsorted(ev_times, t, side="right") - 1] if t >= ev_times[0] else 0.0 for t in times])
    return 1 - np.exp(-np.outer(np.asarray(hr_new, float), H_at))

def rcs_basis(x, knots):
    """Restricted cubic spline basis (Harrell), linear term plus len(knots)-2 non-linear terms."""
    k = len(knots); t = np.asarray(knots, float)
    scale = (t[-1] - t[0]) ** 2
    cols = [x]
    for j in range(k - 2):
        cols.append((np.maximum(x - t[j], 0) ** 3
                     - np.maximum(x - t[k - 2], 0) ** 3 * (t[k - 1] - t[j]) / (t[k - 1] - t[k - 2])
                     + np.maximum(x - t[k - 1], 0) ** 3 * (t[k - 2] - t[j]) / (t[k - 1] - t[k - 2])) / scale)
    return np.column_stack(cols)

def spline_design(train_X, other_Xs, cont_cols, n_knots=4):
    """Replace each continuous column (>= 5 distinct values) by an RCS basis with knots at the 5th, 35th, 65th
    and 95th training percentiles, then standardise all columns on the training data."""
    def expand(X, knots_map):
        parts = [X.drop(columns=list(knots_map))]
        for col, kn in knots_map.items():
            B = rcs_basis(X[col].values.astype(float), kn)
            parts.append(pd.DataFrame(B, index=X.index, columns=[f"{col}__rcs{j}" for j in range(B.shape[1])]))
        return pd.concat(parts, axis=1)
    knots_map = {}
    for col in cont_cols:
        if col in train_X.columns and train_X[col].nunique() >= 5:
            kn = np.unique(np.percentile(train_X[col], [5, 35, 65, 95]))
            if len(kn) >= 3:
                knots_map[col] = kn
    tr = expand(train_X, knots_map)
    sc = StandardScaler().fit(tr)
    std = lambda d: pd.DataFrame(sc.transform(d[tr.columns]), index=d.index, columns=tr.columns)
    return std(tr), [std(expand(X, knots_map)) for X in other_Xs]
print("survival-probability and spline helpers OK")

## Section 4. Load data, rebuild the male cohort, check counts

In [ ]:
# [Cell 14] [Original, modified] read-in step at the start of run_cohort in new code.ipynb
# Change: only the both-sex and female files are read.
DF = {}
for c in ["bothsex", "female"]:
    df = pd.read_csv(COHORT_FILES[c])
    if INVERT_EVENT:
        df[EVENT_COL] = 1 - df[EVENT_COL].astype(int)
    DF[c] = df.dropna(subset=[TIME_COL, EVENT_COL]).reset_index(drop=True)
    print(f"{c}: {DF[c].shape}, events = {int(DF[c][EVENT_COL].sum())}")

In [ ]:
# [Cell 15] [New] Rebuild the complete male cohort from the both-sex file
# Purpose: R1.1, R2.9 (all men in the both-sex file form the male cohort).
sex_values = sorted(DF["bothsex"][SEX_COL].dropna().unique().tolist())
assert len(sex_values) == 2 and MALE_CODE in sex_values, f"unexpected sex coding: {sex_values}"
FEMALE_CODE = [v for v in sex_values if v != MALE_CODE][0]
DF["male"] = DF["bothsex"].loc[DF["bothsex"][SEX_COL] == MALE_CODE].drop(columns=[SEX_COL]).reset_index(drop=True)
print(f"male: {DF['male'].shape}, events = {int(DF['male'][EVENT_COL].sum())}; female code in both-sex file = {FEMALE_CODE}")

In [ ]:
# [Cell 16] [New] Check cohort sizes, deaths and early censoring
# Purpose: R1.1 (stops if counts differ from the expected values).
for c in ["bothsex", "female", "male"]:
    df = DF[c]
    n, d = len(df), int(df[EVENT_COL].sum())
    early = int(((df[EVENT_COL] == 0) & (df[TIME_COL] <= PRIMARY)).sum())
    print(f"{c}: n={n} deaths={d} expected={EXPECTED[c]}  censored before {PRIMARY:.0f} y = {early}")
    if CHECK_EXPECTED_COUNTS and not QUICK_TEST:
        assert (n, d) == EXPECTED[c], f"{c}: counts differ from expected"
    assert early > 0, f"{c}: nobody censored before {PRIMARY} y -> check the input file"
fem_sub = DF["bothsex"][DF["bothsex"][SEX_COL] == FEMALE_CODE]
print("women in both-sex file:", len(fem_sub), "deaths:", int(fem_sub[EVENT_COL].sum()))
if CHECK_EXPECTED_COUNTS and not QUICK_TEST:
    assert (len(fem_sub), int(fem_sub[EVENT_COL].sum())) == EXPECTED["female"], "sex coding check failed"
print("checks passed")

In [ ]:
# [Cell 17] [New] Check the variable names used by the domain models
# Purpose: R1.6 (edit CLINICAL_VARS / DOMAIN_VARS in section 1 if a name is missing or a predictor is misplaced).
cols = set(DF["bothsex"].columns) | set(DF["female"].columns)
print("DEMOGRAPHIC_VARS missing:", [v for v in DEMOGRAPHIC_VARS if v not in cols])
print("CLINICAL_VARS missing:   ", [v for v in CLINICAL_VARS if v not in cols])
for dname, vs in DOMAIN_VARS.items():
    print(f"{dname} missing:", [v for v in vs if v not in cols])
assigned = set(DEMOGRAPHIC_VARS) | set(CLINICAL_VARS) | set(sum(DOMAIN_VARS.values(), []))
print("Predictors added in the last ('remaining') step:",
      sorted(cols - assigned - set(SPECIAL_EXCLUDE)))

## Section 5. Split and preprocess

In [ ]:
# [Cell 18] [Original, unchanged] run_cohort in new code.ipynb: variable groups
BUCKETS = {}
for c in COHORTS_TO_RUN:
    BUCKETS[c] = build_column_buckets(DF[c], BINARY_TXT, MULTI_TXT,
                                      special_exclude=SPECIAL_EXCLUDE,
                                      use_dtype_for_numeric=USE_DTYPE_FOR_NUMERIC)

In [ ]:
# [Cell 19] [Original, unchanged] run_cohort in new code.ipynb: train / calibration / test split
TRAIN, CAL, TEST = {}, {}, {}
for c in COHORTS_TO_RUN:
    df = DF[c]
    stratifier = df[EVENT_COL] if SURVIVAL_STRATIFY else None
    train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=stratifier)
    train_df, cal_df = train_test_split(train_df, test_size=0.20, random_state=SEED, stratify=train_df[EVENT_COL] if SURVIVAL_STRATIFY else None)
    TRAIN[c], CAL[c], TEST[c] = train_df, cal_df, test_df
    print(f" {c}: train={len(train_df)} cal={len(cal_df)} test={len(test_df)}")

In [ ]:
# [Cell 20] [Original, modified] run_cohort in new code.ipynb: unscaled (tree models) and scaled (LR/Cox/SVM) preprocessing
# Change: wrapped in a small function because the same steps are repeated in sections 12-19.
def preprocess_split(train_df, other_dfs, buckets, scale):
    """Fit preprocessing on train_df only and apply it to each frame in other_dfs."""
    b, n, m, _ = buckets
    outs, tr = [], None
    for other in other_dfs:
        tr, ot, art = preprocess_for_model(train_df, other, binary_cols=b, num_cols=n, multi_cat_cols=m,
                                           passthrough_cols=["edate", "survival"], scale_numeric=scale, ranges=RANGES)
        outs.append(ot)
    return tr, outs, art

TR_T, TE_T, CL_T, TR_C, TE_C, CL_C, ART_T = {}, {}, {}, {}, {}, {}, {}
for c in COHORTS_TO_RUN:
    TR_T[c], (TE_T[c], CL_T[c]), ART_T[c] = preprocess_split(TRAIN[c], [TEST[c], CAL[c]], BUCKETS[c], scale=False)
    TR_C[c], (TE_C[c], CL_C[c]), _ = preprocess_split(TRAIN[c], [TEST[c], CAL[c]], BUCKETS[c], scale=True)
    print(f" {c}: processed train {TR_T[c].shape}, test {TE_T[c].shape}, cal {CL_T[c].shape}")
    print("   out-of-range values set to missing in training data (top 5):")
    print(ART_T[c].range_clean_log_["train"].head().to_string(index=False))

In [ ]:
# [Cell 21] [New] Censoring distribution (training set), fixed-horizon labels and weights for test and calibration sets
# Purpose: R1.2, R2.1 ("how the censoring distribution was estimated and on which data").
KM = {c: CensoringKM(TRAIN[c][TIME_COL], TRAIN[c][EVENT_COL]) for c in COHORTS_TO_RUN}
YW, YW_CAL, BOOT = {}, {}, {}
for c in COHORTS_TO_RUN:
    print(f" {c}: G(5)={KM[c].G([5.0])[0]:.3f}  G(10)={KM[c].G([10.0])[0]:.3f}")
    for h in HORIZONS:
        YW[(c, h)] = horizon_outcome(TEST[c][TIME_COL].values, TEST[c][EVENT_COL].values, h, KM[c])
        YW_CAL[(c, h)] = horizon_outcome(CAL[c][TIME_COL].values, CAL[c][EVENT_COL].values, h, KM[c])
        y, w, known = YW[(c, h)]
        BOOT[(c, h)] = boot_idx(horizon_strata(y, known), N_BOOT, seed=SEED + int(h))
        print(f"   {h:.0f} y test: n={len(y)} known={known.sum()} deaths={y[known].sum()} "
              f"censored before={(~known).sum()}")
tick("data ready")

## Section 6. Survival models (one cell per model)

`RISK[(c, m)]` is the test-set risk score (higher = higher risk). `SPROB[(c, m, h)]` and `SPROB_CAL[(c, m, h)]`
are predicted probabilities of death by h on the test and calibration sets (not available for the survival SVM).

In [ ]:
# [Cell 22] [Original, unchanged] new code.ipynb cell 9: survival X/y
XY_T, XY_C, XCAL_T, XCAL_C = {}, {}, {}, {}
for c in COHORTS_TO_RUN:
    XY_T[c] = build_Xy_survival_sksurv(TR_T[c], TE_T[c], TIME_COL, EVENT_COL)
    XY_C[c] = build_Xy_survival_sksurv(TR_C[c], TE_C[c], TIME_COL, EVENT_COL)
    XCAL_T[c] = CL_T[c].drop(columns=[TIME_COL, EVENT_COL])[XY_T[c][0].columns]
    XCAL_C[c] = CL_C[c].drop(columns=[TIME_COL, EVENT_COL])[XY_C[c][0].columns]
RISK, RISK_CAL, SPROB, SPROB_CAL, SURV_MODELS_FIT, CINDEX_ROWS = {}, {}, {}, {}, {}, []

def store_survival(c, name, y_tr, y_te, risk, risk_cal, prob=None, prob_cal=None, model=None):
    ci_ = cindex_auto(y_tr, y_te, risk)
    ci_.update({"cohort": c, "model": name})
    CINDEX_ROWS.append(ci_)
    sign = -1 if ci_["flipped"] else 1
    RISK[(c, name)], RISK_CAL[(c, name)] = sign * np.asarray(risk), sign * np.asarray(risk_cal)
    if prob is not None:
        for k, h in enumerate(HORIZONS):
            SPROB[(c, name, h)], SPROB_CAL[(c, name, h)] = prob[:, k], prob_cal[:, k]
    SURV_MODELS_FIT[(c, name)] = model
    print(f" {c} {name}: Harrell C = {ci_['harrell_c']:.3f}  Uno C = {ci_['uno_c']:.3f}")

In [ ]:
# [Cell 23] [Original, modified] CoxNet from fit_survival_models in new code.ipynb
# Change: also stores test/calibration risk scores and probabilities of death by 5 and 10 years.
for c in COHORTS_TO_RUN:
    X_train_c, X_test_c, y_train_c, y_test_c = XY_C[c]
    coxnet = CoxnetSurvivalAnalysis(l1_ratio=0.5, alphas=None, max_iter=200000, fit_baseline_model=True)
    coxnet.fit(X_train_c, y_train_c)
    store_survival(c, "CoxNet", y_train_c, y_test_c, coxnet.predict(X_test_c), coxnet.predict(XCAL_C[c]),
                   surv_prob_sksurv(coxnet, X_test_c, HORIZONS), surv_prob_sksurv(coxnet, XCAL_C[c], HORIZONS), coxnet)
    print(f"   penalty used (last alpha of the path) = {coxnet.alphas_[-1]:.5f}; "
          f"non-zero coefficients = {int(np.sum(np.abs(coxnet.coef_[:, -1]) > 0))}")

In [ ]:
# [Cell 24] [Original, modified] Weibull AFT from fit_survival_models in new code.ipynb
# Change: as above.
for c in COHORTS_TO_RUN:
    X_train_c, X_test_c, y_train_c, y_test_c = XY_C[c]
    d_tr = X_train_c.copy()
    d_tr["__T__"] = np.asarray(y_train_c["time"]).astype(float)
    d_tr["__E__"] = np.asarray(y_train_c["event"]).astype(int)
    aft = WeibullAFTFitter(penalizer=0.1).fit(d_tr, "__T__", "__E__", show_progress=False)
    prob = 1 - aft.predict_survival_function(X_test_c, times=HORIZONS).T.values
    prob_cal = 1 - aft.predict_survival_function(XCAL_C[c], times=HORIZONS).T.values
    store_survival(c, "WeibullAFT", y_train_c, y_test_c, -aft.predict_median(X_test_c).values,
                   -aft.predict_median(XCAL_C[c]).values, prob, prob_cal, aft)

In [ ]:
# [Cell 25] [Original, modified] RSF from fit_survival_models in new code.ipynb
# Change: as above.
for c in COHORTS_TO_RUN:
    X_train_t, X_test_t, y_train_t, y_test_t = XY_T[c]
    rsf = RandomSurvivalForest(
        n_estimators=500,
        max_depth=10,
        max_leaf_nodes=1000,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=SEED,
        max_features="sqrt",
        n_jobs=-1,
    )
    rsf.fit(X_train_t, y_train_t)
    store_survival(c, "RSF", y_train_t, y_test_t, rsf.predict(X_test_t), rsf.predict(XCAL_T[c]),
                   surv_prob_sksurv(rsf, X_test_t, HORIZONS), surv_prob_sksurv(rsf, XCAL_T[c], HORIZONS), rsf)

In [ ]:
# [Cell 26] [Original, modified] gradient-boosted survival from fit_survival_models in new code.ipynb
# Change: as above.
for c in COHORTS_TO_RUN:
    X_train_t, X_test_t, y_train_t, y_test_t = XY_T[c]
    gbs = GradientBoostingSurvivalAnalysis(
        learning_rate=0.05,
        n_estimators=500,
        max_depth=3,
        random_state=SEED)
    gbs.fit(X_train_t, y_train_t)
    store_survival(c, "GBSurv", y_train_t, y_test_t, gbs.predict(X_test_t), gbs.predict(XCAL_T[c]),
                   surv_prob_sksurv(gbs, X_test_t, HORIZONS), surv_prob_sksurv(gbs, XCAL_T[c], HORIZONS), gbs)

In [ ]:
# [Cell 27] [Original, modified] FastSurvivalSVM from fit_survival_models in new code.ipynb
# Change: as above (ranking model, no probabilities).
for c in COHORTS_TO_RUN:
    X_train_c, X_test_c, y_train_c, y_test_c = XY_C[c]
    fssvm = FastSurvivalSVM(rank_ratio=1.0, max_iter=200, tol=1e-5, random_state=SEED)
    fssvm.fit(X_train_c, y_train_c)
    store_survival(c, "FastSurvSVM", y_train_c, y_test_c, fssvm.predict(X_test_c), fssvm.predict(XCAL_C[c]),
                   model=fssvm)

In [ ]:
# [Cell 28] [Original, modified] XGBoost-Cox from fit_survival_models in new code.ipynb
# Change: as above; probabilities use a Breslow baseline hazard estimated on the training set (R2.6).
for c in COHORTS_TO_RUN:
    X_train_t, X_test_t, y_train_t, y_test_t = XY_T[c]
    t_tr = np.asarray(y_train_t["time"]).astype(float)
    e_tr = np.asarray(y_train_t["event"]).astype(bool)
    y_xgb = np.where(e_tr, t_tr, -t_tr)
    dtr = xgb.DMatrix(np.asarray(X_train_t), label=y_xgb)
    booster = xgb.train({
        "objective": "survival:cox",
        "eval_metric": "cox-nloglik",
        "tree_method": "hist",
        "learning_rate": 0.05,
        "max_depth": 4,
        "seed": SEED},
        dtr, num_boost_round=400)
    hr_tr = booster.predict(dtr)                                  # hazard ratio scale
    hr_te = booster.predict(xgb.DMatrix(np.asarray(X_test_t)))
    hr_cl = booster.predict(xgb.DMatrix(np.asarray(XCAL_T[c])))
    store_survival(c, "XGBoost-Cox", y_train_t, y_test_t, hr_te, hr_cl,
                   breslow_prob(t_tr, e_tr, hr_tr, hr_te, HORIZONS), breslow_prob(t_tr, e_tr, hr_tr, hr_cl, HORIZONS), booster)

### Section 6b. Additional Cox benchmarks (R2.2: ridge and lasso settings; how the Cox penalty was chosen)

The penalty of each model is chosen by Harrell's C on the calibration set (never the test set).
The main elastic-net model above uses the last (smallest) penalty on the path, as in the submitted analysis.

In [ ]:
# [Cell 29] [New] Ridge, lasso and tuned elastic-net Cox
# Purpose: R2.2.
EXTRA_SURV = []
if RUN["extra_cox_benchmarks"]:
    COX_PENALTY_ROWS = []
    for c in COHORTS_TO_RUN:
        X_train_c, X_test_c, y_train_c, y_test_c = XY_C[c]
        ev_cal, t_cal = CL_C[c][EVENT_COL].astype(bool).values, CL_C[c][TIME_COL].astype(float).values
        for name, l1 in [("CoxNet-EN-tuned", 0.5), ("CoxNet-lasso", 1.0)]:
            m = CoxnetSurvivalAnalysis(l1_ratio=l1, alphas=None, max_iter=200000, fit_baseline_model=True).fit(X_train_c, y_train_c)
            cal_c = [concordance_index_censored(ev_cal, t_cal, m.predict(XCAL_C[c], alpha=a))[0] for a in m.alphas_]
            a_best = float(m.alphas_[int(np.argmax(cal_c))])
            store_survival(c, name, y_train_c, y_test_c, m.predict(X_test_c, alpha=a_best), m.predict(XCAL_C[c], alpha=a_best),
                           surv_prob_sksurv(m, X_test_c, HORIZONS, alpha=a_best),
                           surv_prob_sksurv(m, XCAL_C[c], HORIZONS, alpha=a_best), m)
            COX_PENALTY_ROWS.append({"cohort": c, "model": name, "l1_ratio": l1, "alpha_selected": a_best,
                                     "cal_harrell_c": max(cal_c), "alpha_path_min": float(m.alphas_[-1]),
                                     "alpha_path_max": float(m.alphas_[0]),
                                     "n_nonzero": int(np.sum(np.abs(m.coef_[:, int(np.argmax(cal_c))]) > 0))})
        best = None
        for a in [0.1, 1.0, 10.0, 100.0, 1000.0]:
            m = CoxPHSurvivalAnalysis(alpha=a, ties="breslow", n_iter=100).fit(X_train_c, y_train_c)
            cc_ = concordance_index_censored(ev_cal, t_cal, m.predict(XCAL_C[c]))[0]
            COX_PENALTY_ROWS.append({"cohort": c, "model": "Cox-ridge (grid)", "l1_ratio": 0.0, "alpha_selected": a,
                                     "cal_harrell_c": cc_})
            if best is None or cc_ > best[0]:
                best = (cc_, a, m)
        m = best[2]
        store_survival(c, "Cox-ridge", y_train_c, y_test_c, m.predict(X_test_c), m.predict(XCAL_C[c]),
                       surv_prob_sksurv(m, X_test_c, HORIZONS), surv_prob_sksurv(m, XCAL_C[c], HORIZONS), m)
        COX_PENALTY_ROWS.append({"cohort": c, "model": "Cox-ridge", "l1_ratio": 0.0, "alpha_selected": best[1],
                                 "cal_harrell_c": best[0]})
    EXTRA_SURV += ["CoxNet-EN-tuned", "CoxNet-lasso", "Cox-ridge"]
    save_table(pd.DataFrame(COX_PENALTY_ROWS), "eTable4b_cox_penalty_selection")

### Section 6c. Spline benchmarks (Minor 3: non-linear terms within conventional regression)

In [ ]:
# [Cell 30] [New] Elastic-net Cox with restricted cubic splines for continuous predictors
# Purpose: Minor 3 (Cox regression can include non-linear terms); extra benchmark.
SPLINE_X = {}
if RUN["spline_benchmarks"]:
    for c in COHORTS_TO_RUN:
        cont = [col for col in BUCKETS[c][1] if col in TR_T[c].columns and col not in SPECIAL_EXCLUDE]
        feat = lambda d: d.drop(columns=[TIME_COL, EVENT_COL])
        SPLINE_X[c] = spline_design(feat(TR_T[c]), [feat(TE_T[c]), feat(CL_T[c])], cont)
        Xs_tr, (Xs_te, Xs_cl) = SPLINE_X[c]
        y_train_c, y_test_c = XY_C[c][2], XY_C[c][3]
        m = CoxnetSurvivalAnalysis(l1_ratio=0.5, alphas=None, max_iter=200000, fit_baseline_model=True).fit(Xs_tr, y_train_c)
        store_survival(c, "CoxNet-spline", y_train_c, y_test_c, m.predict(Xs_te), m.predict(Xs_cl),
                       surv_prob_sksurv(m, Xs_te, HORIZONS), surv_prob_sksurv(m, Xs_cl, HORIZONS), m)
        print(f"   {c}: spline design {Xs_tr.shape[1]} columns")
    EXTRA_SURV += ["CoxNet-spline"]

In [ ]:
# [Cell 31] [Original, modified] eTable 6a: C-index table (surv_cindex in new code.ipynb cell 9)
# Change: Uno's C added; extra benchmark models included.
surv_cindex = pd.DataFrame(CINDEX_ROWS)
save_table(surv_cindex, "eTable6a_cindex", suppress=False)
display(surv_cindex.round(3))
tick("survival models")

## Section 7. Classification models (one model per horizon)

In [ ]:
# [Cell 32] [Original, modified] eval_classifier_horizon in new code.ipynb (taken out of the function)
# Change: training unchanged (make_horizon_dataset excludes participants censored before the horizon);
#         predictions are made for the full test and calibration sets so IPCW and complete-case
#         evaluations can both be done (R1.2).
PRED, PRED_CAL, CLF_FIT, CLF_COLS = {}, {}, {}, {}
for c in COHORTS_TO_RUN:
    for h in HORIZONS:
        for name in CLF_ORDER:
            use_tree = name in TREE_MODELS
            train_frame = TR_T[c] if use_tree else TR_C[c]
            test_frame = TE_T[c] if use_tree else TE_C[c]
            cal_frame = CL_T[c] if use_tree else CL_C[c]
            Xtr, ytr = make_horizon_dataset(train_frame, h, label_frame=TR_T[c])
            est = new_clf(name, ytr)
            est.fit(Xtr, ytr)
            Xte_full = test_frame.drop(columns=[TIME_COL, EVENT_COL])[Xtr.columns]
            Xcl_full = cal_frame.drop(columns=[TIME_COL, EVENT_COL])[Xtr.columns]
            PRED[(c, name, h)] = est.predict_proba(Xte_full)[:, 1]
            PRED_CAL[(c, name, h)] = est.predict_proba(Xcl_full)[:, 1]
            CLF_FIT[(c, name, h)] = est
            CLF_COLS[(c, name, h)] = list(Xtr.columns)
        print(f" {c} {h:.0f} y: classifiers fitted (training n with known status = {len(ytr)}, deaths = {int(ytr.sum())})")

In [ ]:
# [Cell 33] [New] Logistic regression with restricted cubic splines
# Purpose: Minor 3; extra benchmark.
EXTRA_CLF = []
if RUN["spline_benchmarks"]:
    for c in COHORTS_TO_RUN:
        Xs_tr, (Xs_te, Xs_cl) = SPLINE_X[c]
        for h in HORIZONS:
            _, ytr = make_horizon_dataset(TR_T[c], h)
            keep = horizon_outcome(TRAIN[c][TIME_COL].values, TRAIN[c][EVENT_COL].values, h, KM[c])[2]
            est = new_clf("LogReg", ytr).fit(Xs_tr.loc[keep], ytr)
            PRED[(c, "LogReg-spline", h)] = est.predict_proba(Xs_te)[:, 1]
            PRED_CAL[(c, "LogReg-spline", h)] = est.predict_proba(Xs_cl)[:, 1]
    EXTRA_CLF += ["LogReg-spline"]

In [ ]:
# [Cell 34] [New] Reproduction check: complete-case AUROC should equal the submitted values
# Purpose: confirms that data and environment match the submission (only both-sex and female can be compared;
#          the male cohort differs from the submitted analysis). Prints only, does not stop the run.
for (c, h), sub in SUBMITTED_CC_AUROC.items():
    if c not in COHORTS_TO_RUN:
        continue
    y, w, known = YW[(c, h)]
    cc = roc_auc_score(y[known], PRED[(c, "LightGBM", h)][known])
    flag = "" if abs(cc - sub) < 0.0015 else "  <-- differs"
    print(f" {c} {h:.0f} y: complete-case AUROC {cc:.3f} vs submitted {sub:.3f}{flag}")

In [ ]:
# [Cell 35] [Original, modified] holdout_predictions saving in new code.ipynb
# Change: saves all test-set predictions; row-level, stays in SURE.
for c in COHORTS_TO_RUN:
    out = pd.DataFrame({TIME_COL: TEST[c][TIME_COL].values, EVENT_COL: TEST[c][EVENT_COL].values})
    for (cc_, m, h), s in PRED.items():
        if cc_ == c: out[f"{m}_{h:.0f}y"] = s
    for (cc_, m), s in RISK.items():
        if cc_ == c: out[f"{m}_risk"] = s
    for (cc_, m, h), s in SPROB.items():
        if cc_ == c: out[f"{m}_prob_{h:.0f}y"] = s
    out.to_csv(ROWLEVEL / f"test_predictions_{c}.csv", index=False)
print("row-level predictions saved (do not export)")
tick("classifiers")

## Section 8. Discrimination: eTable 6, paired differences, Figure 2

All models are compared on the same test participants with the same fixed-horizon AUROC
(cumulative/dynamic definition). Survival models are ranked by their risk score; classifiers by their
horizon-specific probability. `LightGBM-10y-model` at 5 years evaluates the 10-year model's scores at 5 years (R2.1).

In [ ]:
# [Cell 36] [New] Score dictionaries used by all later comparisons
def scores_for(c, h):
    """Model -> score used for AUROC on the test set."""
    s = {m: PRED[(c, m, h)] for m in CLF_ORDER + EXTRA_CLF if (c, m, h) in PRED}
    s.update({m: RISK[(c, m)] for m in SURV_NAMES + EXTRA_SURV if (c, m) in RISK})
    if h != PRIMARY and (c, "LightGBM", PRIMARY) in PRED:
        s["LightGBM-10y-model"] = PRED[(c, "LightGBM", PRIMARY)]
    return s

def probs_for(c, h):
    """Model -> (test probability, calibration-set probability) of death by h."""
    p = {m: (PRED[(c, m, h)], PRED_CAL[(c, m, h)]) for m in CLF_ORDER + EXTRA_CLF if (c, m, h) in PRED}
    p.update({m: (SPROB[(c, m, h)], SPROB_CAL[(c, m, h)]) for m in SURV_NAMES + EXTRA_SURV if (c, m, h) in SPROB})
    return p

def family_of(m):
    return "survival" if m in SURV_NAMES + EXTRA_SURV else "classification"

In [ ]:
# [Cell 37] [New] eTable 6b/6c: fixed-horizon AUROC of every model with 95% CI, IPCW and complete case
# Purpose: R1.4, R2.7 (same metric and same participants for all models).
AUC_BOOT, disc_rows = {}, []
for c in COHORTS_TO_RUN:
    ys_tr = Surv.from_arrays(TRAIN[c][EVENT_COL].astype(bool).values, TRAIN[c][TIME_COL].values)
    ys_te = Surv.from_arrays(TEST[c][EVENT_COL].astype(bool).values, TEST[c][TIME_COL].values)
    for h in HORIZONS:
        y, w, known = YW[(c, h)]
        for m, s in scores_for(c, h).items():
            try:
                uno = float(concordance_index_ipcw(ys_tr, ys_te, s, tau=h)[0])
            except Exception:
                uno = np.nan
            for sch in SCHEMES:
                ww = scheme_weights(w, known, sch)
                AUC_BOOT[(c, m, h, sch)] = boot_metric(w_auc, BOOT[(c, h)], y, s, ww)
                lo, hi = ci(AUC_BOOT[(c, m, h, sch)])
                pt = w_auc(y, s, ww)
                disc_rows.append({"cohort": c, "horizon": h, "scheme": sch, "model": m, "family": family_of(m),
                                  "n_test": len(y), "n_known_status": int(known.sum()), "deaths_by_horizon": int(y[known].sum()),
                                  "AUROC": pt, "AUROC_lo": lo, "AUROC_hi": hi, "AUROC_fmt": fmt(pt, (lo, hi)),
                                  "Uno_C_truncated_at_horizon": uno})
disc = pd.DataFrame(disc_rows)
save_table(disc, "eTable6bc_td_auroc_all_models")
display(disc[disc.scheme == PRIMARY_SCHEME].pivot_table(index=["family", "model"], columns=["cohort", "horizon"], values="AUROC").round(3))

In [ ]:
# [Cell 38] [New] Check: own IPCW AUROC and Brier score versus scikit-survival
# Purpose: R1.2 (the estimator is the standard cumulative/dynamic AUROC and the Graf IPCW Brier score).
# Small differences can only arise when censorings are tied with death times (G(T-) vs G(T)).
for c in COHORTS_TO_RUN:
    ys_tr = Surv.from_arrays(TRAIN[c][EVENT_COL].astype(bool).values, TRAIN[c][TIME_COL].values)
    ys_te = Surv.from_arrays(TEST[c][EVENT_COL].astype(bool).values, TEST[c][TIME_COL].values)
    y, w, _ = YW[(c, PRIMARY)]
    for m in ["LightGBM", "CoxNet"]:
        s = PRED[(c, m, PRIMARY)] if m in CLF_ORDER else RISK.get((c, m))
        own = w_auc(y, s, w)
        sk = cumulative_dynamic_auc(ys_tr, ys_te, s, [PRIMARY])[0][0]
        print(f" {c} {m}: AUROC own={own:.4f} sksurv={sk:.4f} diff={abs(own - sk):.5f}")
    p = SPROB[(c, "CoxNet", PRIMARY)]
    own_b = w_brier(y, p, w)
    sk_b = brier_score(ys_tr, ys_te, (1 - p)[:, None], [PRIMARY])[1][0]
    print(f" {c} CoxNet: Brier own={own_b:.4f} sksurv={sk_b:.4f} diff={abs(own_b - sk_b):.5f}")

In [ ]:
# [Cell 39] [New] eTable 6e: paired bootstrap AUROC differences, LightGBM minus each model
# Purpose: R1.4, R2.7 ("paired confidence intervals for performance differences").
diff_rows = []
for (c, m, h, sch), bootv in AUC_BOOT.items():
    if m == "LightGBM":
        continue
    ref = disc.query("cohort == @c and horizon == @h and scheme == @sch")
    point = ref.loc[ref.model == "LightGBM", "AUROC"].iloc[0] - ref.loc[ref.model == m, "AUROC"].iloc[0]
    d = AUC_BOOT[(c, "LightGBM", h, sch)] - bootv
    lo, hi = ci(d)
    diff_rows.append({"cohort": c, "horizon": h, "scheme": sch, "comparison": f"LightGBM - {m}",
                      "difference": point, "lo": lo, "hi": hi, "fmt": fmt(point, (lo, hi)),
                      "boot_share_LightGBM_better": float(np.nanmean(d > 0))})
diffs = pd.DataFrame(diff_rows)
save_table(diffs, "eTable6e_paired_auroc_differences", suppress=False)
display(diffs[(diffs.horizon == PRIMARY) & (diffs.scheme == PRIMARY_SCHEME)].round(3))

In [ ]:
# [Cell 40] [New] Figure 2: 10-year AUROC of the 12 original models (one figure per scheme) and of all models
# Purpose: R1.4 ("Figure 2 should emphasize common-horizon AUROCs").
def auroc_dot_plot(d10, order, name, xlabel):
    fig, ax = plt.subplots(figsize=(6.7, 0.32 * len(order) + 1.2), layout="constrained")
    k_off = {c: (k - (len(COHORTS_TO_RUN) - 1) / 2) * 0.22 for k, c in enumerate(COHORTS_TO_RUN)}
    for c in COHORTS_TO_RUN:
        sub = d10[d10.cohort == c].set_index("model").reindex(order)
        ypos = np.arange(len(order)) + k_off[c]
        err = np.clip(np.vstack([sub["AUROC"] - sub["AUROC_lo"], sub["AUROC_hi"] - sub["AUROC"]]), 0, None)
        ax.errorbar(sub["AUROC"], ypos, xerr=err, fmt="o", ms=3.5, lw=0.9, capsize=1.5,
                    color=COHORT_COLOR[c], label=COHORT_LABEL[c])
    n_surv = sum(family_of(m) == "survival" for m in order)
    ax.axhline(n_surv - 0.5, color="grey", lw=0.6, ls=":")
    ax.set_yticks(np.arange(len(order)))
    ax.set_yticklabels([MODEL_LABEL.get(m, m) for m in order])
    ax.set_ylim(len(order) - 0.5, -0.9)
    tr_ = matplotlib.transforms.blended_transform_factory(ax.transAxes, ax.transData)
    ax.text(0.01, -0.6, "Survival models", transform=tr_, fontsize=8, color="grey")
    ax.text(0.01, n_surv - 0.6, "Classification models", transform=tr_, fontsize=8, color="grey")
    ax.set_xlabel(xlabel)
    ax.legend(frameon=False, loc="lower left")
    ax.grid(axis="x", lw=0.3, alpha=0.5)
    save_fig(fig, name)

for sch in SCHEMES:
    d10 = disc[(disc.horizon == PRIMARY) & (disc.scheme == sch)]
    lab = "IPCW" if sch == "ipcw" else "complete case"
    auroc_dot_plot(d10, SURV_NAMES + CLF_ORDER, f"Figure2_td_auroc_10y_{sch}", f"10-year AUROC, {lab} (95% CI)")
    if EXTRA_SURV or EXTRA_CLF:
        auroc_dot_plot(d10, SURV_NAMES + EXTRA_SURV + CLF_ORDER + EXTRA_CLF, f"eFigure_td_auroc_10y_all_models_{sch}",
                       f"10-year AUROC, {lab} (95% CI)")

In [ ]:
# [Cell 41] [New] Time-dependent AUROC across follow-up (survival-model risk scores and the 10-year LightGBM score)
# Purpose: extra analysis supporting R1.4 / R2.7 (comparison at more than two horizons).
td_rows = []
for c in COHORTS_TO_RUN:
    t_max = min(TRAIN[c][TIME_COL].max(), TEST[c][TIME_COL].max())
    grid = [t for t in np.arange(1.0, 14.0, 1.0) if t < t_max - 0.25]
    scores = {m: RISK[(c, m)] for m in SURV_NAMES if (c, m) in RISK}
    scores["LightGBM (10-year model)"] = PRED[(c, "LightGBM", PRIMARY)]
    for t in grid:
        y, w, known = horizon_outcome(TEST[c][TIME_COL].values, TEST[c][EVENT_COL].values, t, KM[c])
        if y[known].sum() < 10 or (known & (y == 0)).sum() < 10:
            continue
        for m, s in scores.items():
            td_rows.append({"cohort": c, "time": t, "model": m, "AUROC_ipcw": w_auc(y, s, w),
                            "deaths_by_time": int(y[known].sum())})
td_auc = pd.DataFrame(td_rows)
save_table(td_auc, "eTable6g_td_auroc_over_time")
fig, axes = plt.subplots(1, len(COHORTS_TO_RUN), figsize=(6.7, 2.6), sharey=True, squeeze=False, layout="constrained")
for j, c in enumerate(COHORTS_TO_RUN):
    ax = axes[0, j]
    for m, g in td_auc[td_auc.cohort == c].groupby("model"):
        ax.plot(g["time"], g["AUROC_ipcw"], lw=1.4 if "LightGBM" in m else 0.9,
                color="#D55E00" if "LightGBM" in m else None, label=MODEL_LABEL.get(m, m))
    ax.set_title(COHORT_LABEL[c]); ax.set_xlabel("Years since enrolment")
axes[0, 0].set_ylabel("Time-dependent AUROC (IPCW)")
handles, labels_ = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="outside lower center", ncol=4, frameon=False)
save_fig(fig, "eFigure_td_auroc_over_time")
tick("discrimination")

## Section 9. Calibration, recalibration and clinical utility

Recalibration (Platt and isotonic) is fitted on the calibration set with the weights of the same scheme.
`VERSIONS[(c, h, scheme)]` holds the original, Platt and isotonic test-set probabilities of LightGBM.

In [ ]:
# [Cell 42] [New] Platt and isotonic recalibration of LightGBM on the calibration set
# Purpose: R1.3, R2.6 (before and after recalibration; identify the prediction version).
VERSIONS, PLATT = {}, {}
for c in COHORTS_TO_RUN:
    for h in HORIZONS:
        yc, wc, kc = YW_CAL[(c, h)]
        p_raw, pc_raw = PRED[(c, "LightGBM", h)], PRED_CAL[(c, "LightGBM", h)]
        for sch in SCHEMES:
            wcs = scheme_weights(wc, kc, sch)
            PLATT[(c, h, sch)] = Platt().fit(pc_raw, yc, wcs)
            iso = Isotonic().fit(pc_raw, yc, wcs)
            VERSIONS[(c, h, sch)] = {"original": p_raw, "Platt": PLATT[(c, h, sch)](p_raw), "isotonic": iso(p_raw)}
            print(f" {c} {h:.0f} y {sch}: Platt a={PLATT[(c, h, sch)].a:.3f} b={PLATT[(c, h, sch)].b:.3f}")

In [ ]:
# [Cell 43] [New] Table 1 / eTable 6d: LightGBM discrimination, overall performance and calibration by version and scheme
# Purpose: R1.3, R2.6 (intercepts, slopes and Brier scores before and after recalibration, with 95% CIs).
cal_rows = []
for c in COHORTS_TO_RUN:
    for h in HORIZONS:
        y, w, known = YW[(c, h)]
        for sch in SCHEMES:
            ww = scheme_weights(w, known, sch)
            for v, p in VERSIONS[(c, h, sch)].items():
                row = {"cohort": c, "horizon": h, "scheme": sch, "version": v, "n_test": len(y),
                       "n_known_status": int(known.sum()), "deaths_by_horizon": int(y[known].sum()),
                       "censored_before_horizon": int((~known).sum()),
                       "platt_a": PLATT[(c, h, sch)].a if v == "Platt" else np.nan,
                       "platt_b": PLATT[(c, h, sch)].b if v == "Platt" else np.nan,
                       "mean_predicted": float(np.average(p[ww > 0], weights=ww[ww > 0])),
                       "observed_risk": w_prevalence(y, ww)}
                row.update(metric_row(y, p, ww, BOOT[(c, h)], list(METRICS)))
                cal_rows.append(row)
calib = pd.DataFrame(cal_rows)
save_table(calib, "Table1_eTable6d_lightgbm_calibration")
display(calib[(calib.horizon == PRIMARY)][["cohort", "scheme", "version", "AUROC_fmt", "Brier_fmt",
                                            "cal_intercept_fmt", "cal_slope_fmt"]])

In [ ]:
# [Cell 44] [New] eTable 6f: calibration and Brier score of every model that gives probabilities, original and Platt,
#       with paired Brier differences versus LightGBM
# Purpose: R1.3, R1.4, R2.6 (calibration of the comparison models; survival-model probabilities, R2.6).
allcal_rows, BRIER_BOOT = [], {}
for c in COHORTS_TO_RUN:
    for h in HORIZONS:
        y, w, known = YW[(c, h)]
        yc, wc, kc = YW_CAL[(c, h)]
        idx = BOOT[(c, h)][:N_BOOT_SECONDARY]
        for sch in SCHEMES:
            ww, wcs = scheme_weights(w, known, sch), scheme_weights(wc, kc, sch)
            for m, (p, pc) in probs_for(c, h).items():
                versions = {"original": p, "Platt": Platt().fit(pc, yc, wcs)(p)}
                for v, pv in versions.items():
                    row = {"cohort": c, "horizon": h, "scheme": sch, "model": m, "family": family_of(m), "version": v}
                    row.update(metric_row(y, pv, ww, idx, ["Brier", "scaled_Brier", "cal_intercept", "cal_slope", "OE_ratio"]))
                    allcal_rows.append(row)
                    BRIER_BOOT[(c, m, h, sch, v)] = boot_metric(w_brier, idx, y, pv, ww)
allcal = pd.DataFrame(allcal_rows)
save_table(allcal, "eTable6f_calibration_all_models", suppress=False)
bd_rows = []
for (c, m, h, sch, v), bv in BRIER_BOOT.items():
    if m == "LightGBM":
        continue
    ref = allcal.query("cohort == @c and horizon == @h and scheme == @sch and version == @v").set_index("model")["Brier"]
    lo, hi = ci(BRIER_BOOT[(c, "LightGBM", h, sch, v)] - bv)
    pt = ref["LightGBM"] - ref[m]
    bd_rows.append({"cohort": c, "horizon": h, "scheme": sch, "version": v, "comparison": f"LightGBM - {m}",
                    "Brier_difference": pt, "lo": lo, "hi": hi, "fmt": fmt(pt, (lo, hi), 4)})
brier_diffs = pd.DataFrame(bd_rows)
save_table(brier_diffs, "eTable6e2_paired_brier_differences", suppress=False)
display(allcal[(allcal.horizon == PRIMARY) & (allcal.scheme == PRIMARY_SCHEME)].pivot_table(
    index="model", columns=["cohort", "version"], values="Brier").round(4))

In [ ]:
# [Cell 45] [New] Reliability tables (deciles) with bootstrap 95% CI of the observed risk
# Purpose: R1.2 (CIs in eFigure 3 and how censoring was handled), R2.6.
rel_rows = []
for c in COHORTS_TO_RUN:
    for h in HORIZONS:
        y, w, known = YW[(c, h)]
        idx = BOOT[(c, h)][:max(N_BOOT_SECONDARY, min(N_BOOT, 500))]
        for sch in SCHEMES:
            ww = scheme_weights(w, known, sch)
            for v in ["original", "Platt", "isotonic"]:
                p = VERSIONS[(c, h, sch)][v]
                rt, edges = reliability_table_w(y, p, ww)
                obs = np.full((len(idx), len(rt)), np.nan)
                for b, ii in enumerate(idx):
                    rb, _ = reliability_table_w(y[ii], p[ii], ww[ii], edges=edges)
                    obs[b] = rb.set_index("bin").reindex(rt["bin"])["obs"].values
                rt["obs_lo"], rt["obs_hi"] = np.nanpercentile(obs, 2.5, axis=0), np.nanpercentile(obs, 97.5, axis=0)
                for k_, v_ in {"version": v, "scheme": sch, "horizon": h, "cohort": c}.items():
                    rt.insert(0, k_, v_)
                rel_rows.append(rt)
rel = pd.concat(rel_rows, ignore_index=True)
save_table(rel, "eFigure1_3_reliability_values")

In [ ]:
# [Cell 46] [New] eTable 7 / eTable 9: operating characteristics and confusion matrices (original and Platt probabilities)
# Purpose: R1.1, R2.9 (replace eTables 7a and 9; each row sums to its evaluation sample).
# Thresholds: Youden on the test set (as submitted), Youden on the calibration set (not test-derived), 7.5%, 20%.
op_rows = []
for c in COHORTS_TO_RUN:
    for h in HORIZONS:
        y, w, known = YW[(c, h)]
        yc, wc, kc = YW_CAL[(c, h)]
        for sch in SCHEMES:
            ww, wcs = scheme_weights(w, known, sch), scheme_weights(wc, kc, sch)
            for v in ["original", "Platt"]:
                p = VERSIONS[(c, h, sch)][v]
                pc = PRED_CAL[(c, "LightGBM", h)] if v == "original" else PLATT[(c, h, sch)](PRED_CAL[(c, "LightGBM", h)])
                thr = {"Youden (test set)": youden_threshold_w(y, p, ww),
                       "Youden (calibration set)": youden_threshold_w(yc, pc, wcs), **OP_THRESHOLDS}
                for lab, t in thr.items():
                    op_rows.append({"cohort": c, "horizon": h, "scheme": sch, "version": v, "threshold_label": lab,
                                    "n_known_status": int(known.sum()), "deaths_by_horizon": int(y[known].sum()),
                                    **operating_point(y, p, ww, t, known)})
ops = pd.DataFrame(op_rows)
save_table(ops, "eTable7_eTable9_operating_characteristics")
display(ops[(ops.horizon == PRIMARY) & (ops.scheme == "cc") & (ops.version == "original")].round(3))

In [ ]:
# [Cell 47] [New] Decision curves for LightGBM (original and Platt) and every model with probabilities (Platt)
# Purpose: R1.2 (censoring in net benefit), Minor 2 (threshold range).
dca_rows, DCA_RANGE = [], {}
for c in COHORTS_TO_RUN:
    for h in HORIZONS:
        y, w, known = YW[(c, h)]
        yc, wc, kc = YW_CAL[(c, h)]
        for sch in SCHEMES:
            ww, wcs = scheme_weights(w, known, sch), scheme_weights(wc, kc, sch)
            curves = {"LightGBM|original": VERSIONS[(c, h, sch)]["original"], "LightGBM|Platt": VERSIONS[(c, h, sch)]["Platt"]}
            for m, (p, pc) in probs_for(c, h).items():
                if m != "LightGBM":
                    curves[f"{m}|Platt"] = Platt().fit(pc, yc, wcs)(p)
            ta = np.array([treat_all_nb(y, ww, t) for t in DCA_THRESHOLDS])
            for lab, p in curves.items():
                nb = np.array([net_benefit(y, p, ww, t) for t in DCA_THRESHOLDS])
                dca_rows.append(pd.DataFrame({"cohort": c, "horizon": h, "scheme": sch, "model": lab,
                                              "threshold": DCA_THRESHOLDS, "net_benefit": nb,
                                              "treat_all": ta, "treat_none": 0.0}))
                ok = nb > np.maximum(ta, 0) + 1e-9
                DCA_RANGE[(c, h, sch, lab)] = (float(DCA_THRESHOLDS[ok].min()), float(DCA_THRESHOLDS[ok].max())) if ok.any() else (np.nan, np.nan)
dca = pd.concat(dca_rows, ignore_index=True)
save_table(dca, "eFigure5_dca_values", suppress=False)
dca_range = pd.DataFrame([{"cohort": k[0], "horizon": k[1], "scheme": k[2], "model": k[3],
                           "useful_from": v[0], "useful_to": v[1]} for k, v in DCA_RANGE.items()])
save_table(dca_range, "eFigure5_dca_useful_threshold_range", suppress=False)
display(dca_range[(dca_range.model.str.startswith("LightGBM")) & (dca_range.horizon == PRIMARY)])
tick("calibration")

## Section 10. Supplementary figures (one file per scheme / version)

In [ ]:
# [Cell 48] [New] eFigure 1: LightGBM reliability curves, original vs Platt, by cohort and horizon
# Purpose: R1.3, R2.6.
for sch in SCHEMES:
    fig, axes = plt.subplots(2, len(COHORTS_TO_RUN), figsize=(6.7, 4.8), sharex=True, sharey=True,
                             squeeze=False, layout="constrained")
    for i, h in enumerate(HORIZONS):
        for j, c in enumerate(COHORTS_TO_RUN):
            ax = axes[i, j]
            ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=0.8)
            for v, col in [("original", "#999999"), ("Platt", "#D55E00")]:
                r = rel[(rel.cohort == c) & (rel.horizon == h) & (rel.version == v) & (rel.scheme == sch)]
                ax.plot(r["pred_mean"], r["obs"], "o-", ms=3, lw=1, color=col, label=v)
            ax.set_title(f"{COHORT_LABEL[c]}, {h:.0f} y")
            if i == 1: ax.set_xlabel("Predicted risk")
            if j == 0: ax.set_ylabel("Observed risk" + (" (IPCW)" if sch == "ipcw" else ""))
    axes[0, 0].legend(frameon=False)
    save_fig(fig, f"eFigure1_reliability_lightgbm_{sch}")

In [ ]:
# [Cell 49] [Original, modified] distribution plot from the error-analysis cell of new code.ipynb
# Change: fixes the bug (the original read each cohort's file into hp_path but plotted the leftover male hp,
#         so every panel showed men); each panel uses its own cohort's predictions; only participants with
#         known status at the horizon are shown, so n matches eTable 7 (R1.1, R2.9).
for v in ["original", "Platt"]:
    fig, axes = plt.subplots(2, len(COHORTS_TO_RUN), figsize=(6.7, 4.6), sharex=True, squeeze=False, layout="constrained")
    for i, h in enumerate(HORIZONS):
        for j, c in enumerate(COHORTS_TO_RUN):
            y, w, known = YW[(c, h)]
            p = VERSIONS[(c, h, PRIMARY_SCHEME)][v]
            ax, bins = axes[i, j], np.linspace(0, 1, 31)
            ax.hist(p[known & (y == 0)], bins=bins, density=True, alpha=0.55, color="#0072B2",
                    label=f"Alive (n={int((known & (y == 0)).sum()):,})")
            ax.hist(p[known & (y == 1)], bins=bins, density=True, alpha=0.55, color="#D55E00",
                    label=f"Died (n={int((known & (y == 1)).sum()):,})")
            ax.set_title(f"{COHORT_LABEL[c]}, {h:.0f} y")
            ax.legend(frameon=False, loc="upper right", handlelength=1)
            if i == 1: ax.set_xlabel(f"Predicted risk ({v})")
            if j == 0: ax.set_ylabel("Density")
    save_fig(fig, f"eFigure2_risk_distributions_{v}")

In [ ]:
# [Cell 50] [New] eFigure 3: 10-year decile calibration with bootstrap 95% CI
# Purpose: R1.2 (CIs in eFigure 3), R1.3.
for sch in SCHEMES:
    fig, axes = plt.subplots(1, len(COHORTS_TO_RUN), figsize=(6.7, 2.6), sharey=True, squeeze=False, layout="constrained")
    for j, c in enumerate(COHORTS_TO_RUN):
        ax = axes[0, j]
        ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=0.8)
        for v, col in [("original", "#999999"), ("Platt", "#D55E00")]:
            r = rel[(rel.cohort == c) & (rel.horizon == PRIMARY) & (rel.version == v) & (rel.scheme == sch)]
            err = np.clip(np.vstack([r["obs"] - r["obs_lo"], r["obs_hi"] - r["obs"]]), 0, None)
            ax.errorbar(r["pred_mean"], r["obs"], yerr=err, fmt="o-", ms=3, lw=0.9, capsize=1.5, color=col, label=v)
        ax.set_title(COHORT_LABEL[c]); ax.set_xlabel("Predicted 10-year risk")
    axes[0, 0].set_ylabel("Observed risk" + (" (IPCW)" if sch == "ipcw" else ""))
    axes[0, 0].legend(frameon=False)
    save_fig(fig, f"eFigure3_decile_calibration_10y_{sch}")

In [ ]:
# [Cell 51] [Original, modified] reliability_plot (all classifiers) in new code.ipynb
# Change: observed risk by scheme; XGBoost labelled "XGBoost (classifier)" (R2.6).
palette = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00", "#CC79A7", "#0072B2"]
for sch in SCHEMES:
    fig, axes = plt.subplots(1, len(COHORTS_TO_RUN), figsize=(6.7, 3.0), sharey=True, squeeze=False, layout="constrained")
    for j, c in enumerate(COHORTS_TO_RUN):
        y, w, known = YW[(c, PRIMARY)]
        ax = axes[0, j]
        ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=0.8)
        for m, col in zip(CLF_ORDER, palette):
            r, _ = reliability_table_w(y, PRED[(c, m, PRIMARY)], scheme_weights(w, known, sch))
            ax.plot(r["pred_mean"], r["obs"], "o-", ms=2.5, lw=0.9, color=col, label=MODEL_LABEL[m])
        ax.set_title(COHORT_LABEL[c]); ax.set_xlabel("Predicted 10-year risk")
    axes[0, 0].set_ylabel("Observed risk" + (" (IPCW)" if sch == "ipcw" else ""))
    handles, labels_ = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels_, loc="outside lower center", ncol=3, frameon=False)
    save_fig(fig, f"eFigure4_reliability_all_classifiers_10y_{sch}")

In [ ]:
# [Cell 52] [Original, modified] dca_plot in new code.ipynb
# Change: LightGBM original and Platt, weighted by scheme, three cohorts in one figure (R1.2);
#         a second figure compares models after Platt recalibration.
for sch in SCHEMES:
    for name, curves in [("eFigure5_dca_10y", [("LightGBM|original", "#999999", "LightGBM (original)"),
                                               ("LightGBM|Platt", "#D55E00", "LightGBM (Platt)")]),
                         ("eFigure5b_dca_10y_models", [("LightGBM|Platt", "#D55E00", "LightGBM"),
                                                       ("LogReg|Platt", "#0072B2", "Logistic regression"),
                                                       ("CoxNet|Platt", "#009E73", "Penalized Cox")])]:
        fig, axes = plt.subplots(1, len(COHORTS_TO_RUN), figsize=(6.7, 2.8), sharey=True, squeeze=False, layout="constrained")
        for j, c in enumerate(COHORTS_TO_RUN):
            dc = dca[(dca.cohort == c) & (dca.horizon == PRIMARY) & (dca.scheme == sch)]
            ax = axes[0, j]
            for lab, col, txt in curves:
                g = dc[dc.model == lab]
                ax.plot(g["threshold"], g["net_benefit"], color=col, lw=1.2, label=txt)
            g = dc[dc.model == curves[0][0]]
            ax.plot(g["threshold"], g["treat_all"], color="grey", lw=1, ls="--", label="Treat all")
            ax.axhline(0, color="black", lw=0.8, label="Treat none")
            ax.set_ylim(-0.05, max(0.05, dc["net_benefit"].max() * 1.1))
            ax.set_title(COHORT_LABEL[c]); ax.set_xlabel("Risk threshold")
        axes[0, 0].set_ylabel("Net benefit")
        handles, labels_ = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels_, loc="outside lower center", ncol=len(labels_), frameon=False)
        save_fig(fig, f"{name}_{sch}")
tick("figures")

## Section 11. SHAP (LightGBM) and descriptive tables

In [ ]:
# [Cell 53] [Original, modified] SHAP computation at the end of run_cohort in new code.ipynb
# Change: uses the 10-year model (primary_h = HORIZONS[0] in the original was 5 years; submitted Figure 3 came
#         from the ever-died model) and the full test set; the 5-year model is also explained.
#         TreeExplainer without data: tree-path-dependent, no background data, log-odds scale (R2.8).
SHAP_SV, SHAP_X, SHAP_TABLE = {}, {}, {}
for c in COHORTS_TO_RUN:
    for h in HORIZONS:
        Xte = TE_T[c].drop(columns=[TIME_COL, EVENT_COL])[CLF_COLS[(c, "LightGBM", h)]]
        sv = shap_matrix(CLF_FIT[(c, "LightGBM", h)], Xte)
        SHAP_SV[(c, h)], SHAP_X[(c, h)] = sv, Xte
        print(f" {c} {h:.0f} y: SHAP {sv.shape}")

In [ ]:
# [Cell 54] [Original, modified] SHAP direction from Survival_for_data_colloection.ipynb (binary: mean difference;
#                      continuous: correlation)
# Change: applied to both horizons; ranking table saved.
for (c, h), sv in SHAP_SV.items():
    X = SHAP_X[(c, h)]
    directions = [shap_direction(X[col].values, sv[:, j]) for j, col in enumerate(X.columns)]
    tab = pd.DataFrame({"feature": X.columns, "label": [LABELS.get(f, f) for f in X.columns],
                        "mean_abs_shap": np.abs(sv).mean(0), "direction": directions})
    tab["higher_value_associated_with"] = [direction_label(d) for d in tab["direction"]]
    tab = tab.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    tab["rank"] = tab["mean_abs_shap"].rank(ascending=False, method="min").astype(int)
    save_table(tab, f"Figure3_shap_{c}_{h:.0f}y", suppress=False)
    SHAP_TABLE[(c, h)] = tab
display(SHAP_TABLE[(COHORTS_TO_RUN[0], PRIMARY)].head(20).round(3))

In [ ]:
# [Cell 55] [New] eTable 12: grouped SHAP importance of the original categorical variables
# Purpose: R2.8 ("a clearly defined grouped summary for the original categorical variables").
# SHAP values are additive, so the attribution of a variable is the sum over its indicators for each person.
group_rows = []
for (c, h), sv in SHAP_SV.items():
    multi_cols = BUCKETS[c][2]
    X, tab = SHAP_X[(c, h)], SHAP_TABLE[(c, h)]
    parents = [parent_of(col, multi_cols) for col in X.columns]
    grouped = pd.DataFrame(sv, columns=X.columns).T.groupby(parents).sum().T
    g = pd.DataFrame({"variable": grouped.columns, "grouped_mean_abs_shap": np.abs(grouped.values).mean(0)})
    g = g.sort_values("grouped_mean_abs_shap", ascending=False).reset_index(drop=True)
    g["grouped_rank"] = np.arange(1, len(g) + 1)
    best = tab.assign(parent=[parent_of(f, multi_cols) for f in tab["feature"]]).groupby("parent")["rank"].min()
    g["best_indicator_rank"] = g["variable"].map(best)
    g.insert(0, "horizon", h); g.insert(0, "cohort", c)
    group_rows.append(g)
grouped_shap = pd.concat(group_rows, ignore_index=True)
save_table(grouped_shap, "eTable12_grouped_shap", suppress=False)
display(grouped_shap[grouped_shap.horizon == PRIMARY].groupby("cohort").head(10))

In [ ]:
# [Cell 56] [Original, modified] SHAP figures (bar + beeswarm) from Survival_for_data_colloection.ipynb
# Change: 10-year model; drawn at the final 170-mm width so fonts stay at 8 pt; labels from LABELS;
#         max_display=21 = top 20 predictors + "Sum of other features" (the submitted figure showed 19);
#         panel letters follow the Figure 3 legend (A both sexes, B males, C females).
for k, c in enumerate(COHORTS_TO_RUN):
    sv, X = SHAP_SV[(c, PRIMARY)], SHAP_X[(c, PRIMARY)]
    exp = shap.Explanation(values=sv, data=X.values, feature_names=[LABELS.get(f, f) for f in X.columns])
    fig = plt.figure(figsize=(6.7, 5.0))
    ax_bar = fig.add_axes([0.37, 0.09, 0.22, 0.89])          # left margin holds the variable names
    ax_swarm = fig.add_axes([0.61, 0.09, 0.30, 0.89])        # SHAP adds its colour bar to the right
    shap.plots.bar(exp, max_display=21, show=False, ax=ax_bar)
    ax_bar.set_xlim(0, ax_bar.get_xlim()[1] * 1.25)
    shap.plots.beeswarm(exp, max_display=21, show=False, ax=ax_swarm, plot_size=None)
    ax_swarm.set_yticklabels([]); ax_swarm.set_ylabel("")
    ax_bar.set_xlabel("Mean |SHAP value|")
    ax_swarm.set_xlabel("SHAP value (log-odds)")
    for ax in (ax_bar, ax_swarm):
        for t in ax.get_yticklabels() + ax.get_xticklabels() + [ax.xaxis.label]:
            t.set_fontsize(8)
        for t in ax.texts:
            t.set_fontsize(8)
    for cax in fig.axes[2:]:
        cax.tick_params(labelsize=8); cax.yaxis.label.set_fontsize(8)
    save_fig(fig, f"Figure3{ {'bothsex': 'A', 'male': 'B', 'female': 'C'}[c] }_shap_{c}_10y")

In [ ]:
# [Cell 57] [New] Binned SHAP dependence for the top predictors (aggregate values only, suitable for export)
# Purpose: extra analysis for R2.8 (shape and direction of the attributions, e.g. hearing loss).
dep_rows = []
for c in COHORTS_TO_RUN:
    sv, X, tab = SHAP_SV[(c, PRIMARY)], SHAP_X[(c, PRIMARY)], SHAP_TABLE[(c, PRIMARY)]
    for f in list(tab["feature"].head(12)) + [k for k in KEY_PREDICTORS if k in X.columns and k not in tab["feature"].head(12).values]:
        x, s = X[f].values, sv[:, list(X.columns).index(f)]
        if len(np.unique(x)) <= 10:
            bins = pd.Series(x).astype(float)
        else:
            bins = pd.Series(pd.qcut(x, 10, duplicates="drop")).cat.codes
        g = pd.DataFrame({"bin": bins, "x": x, "s": s}).groupby("bin").agg(x_mean=("x", "mean"), x_min=("x", "min"),
                                                                           x_max=("x", "max"), shap_mean=("s", "mean"),
                                                                           shap_sd=("s", "std"), n=("s", "size")).reset_index()
        g.insert(0, "feature", f); g.insert(0, "cohort", c)
        dep_rows.append(g)
dep = pd.concat(dep_rows, ignore_index=True)
save_table(dep, "eTable13_binned_shap_dependence")
for c in COHORTS_TO_RUN:
    feats = SHAP_TABLE[(c, PRIMARY)]["feature"].head(12).tolist()
    fig, axes = plt.subplots(3, 4, figsize=(6.7, 5.4), squeeze=False, layout="constrained")
    for ax, f in zip(axes.ravel(), feats):
        g = dep[(dep.cohort == c) & (dep.feature == f)]
        ax.plot(g["x_mean"], g["shap_mean"], "o-", ms=3, lw=1, color=COHORT_COLOR[c])
        ax.axhline(0, color="grey", lw=0.6, ls=":")
        ax.set_title(LABELS.get(f, f)[:32])
    fig.supylabel("Mean SHAP value (log-odds)")
    save_fig(fig, f"eFigure_shap_dependence_{c}_10y")

In [ ]:
# [Cell 58] [Original, modified] STROBE-flow cell in new code.ipynb (eTable 1)
# Change: split sizes are taken from the actual split (the original estimated them with round(), giving 2,334
#         men); age range, age groups and censoring before each horizon added (Minor 1, R1.2).
e1 = []
for c in COHORTS_TO_RUN:
    df = DF[c]
    t, e = df[TIME_COL].astype(float), df[EVENT_COL]
    a = pd.to_numeric(df[AGE_COL], errors="coerce")
    n_pred = len([x for x in sum(BUCKETS[c][:3], []) if x not in SPECIAL_EXCLUDE])
    row = {"cohort": c, "n": len(df), "deaths": int(e.sum()), "deaths_pct": 100 * e.mean(),
           "person_years": float(t.sum()), "mortality_per_1000_py": 1000 * e.sum() / t.sum(),
           "followup_median": t.median(), "followup_q1": t.quantile(.25), "followup_q3": t.quantile(.75),
           "followup_max": t.max(), "age_mean": a.mean(), "age_sd": a.std(), "age_median": a.median(),
           "age_q1": a.quantile(.25), "age_q3": a.quantile(.75), "age_min": a.min(), "age_max": a.max(),
           "n_train": len(TRAIN[c]), "n_cal": len(CAL[c]), "n_test": len(TEST[c]),
           "n_predictors": n_pred, "n_model_columns": TR_T[c].shape[1] - 2, "EPV": e.sum() / n_pred}
    for h in HORIZONS:
        row[f"censored_before_{h:.0f}y"] = int(((e == 0) & (t <= h)).sum())
        row[f"deaths_by_{h:.0f}y"] = int(((e == 1) & (t <= h)).sum())
        row[f"alive_at_{h:.0f}y"] = int((t > h).sum())
    bands = pd.cut(a, AGE_BANDS, right=False, labels=AGE_BAND_LABELS)
    for lab, n in bands.value_counts().reindex(bands.cat.categories).items():
        row[f"age_{lab}_n"], row[f"age_{lab}_pct"] = int(n), 100 * n / len(df)
    e1.append(row)
etable1 = pd.DataFrame(e1)
save_table(etable1, "eTable1_cohort_followup_age")
display(etable1.T)

In [ ]:
# [Cell 59] [New] eTable 3: missingness by sex, plausible ranges, out-of-range values and 99999 codes
# Purpose: R1.8, R2.5, Minor 5.
e3 = []
for c in COHORTS_TO_RUN:
    df = DF[c]
    bin_cols, num_cols, multi_cols, _ = BUCKETS[c]
    for col in [x for x in bin_cols + num_cols + multi_cols if x not in SPECIAL_EXCLUDE]:
        s = df[col]
        n_oor = 0
        if col in RANGES:
            v = pd.to_numeric(s, errors="coerce")
            n_oor = int((v.notna() & ((v < RANGES[col][0]) | (v > RANGES[col][1]))).sum())
        n_code = int(s.isin(MULTI_NA_CODES).sum()) if col in multi_cols else 0
        e3.append({"cohort": c, "variable": col,
                   "type": "binary" if col in bin_cols else ("continuous" if col in num_cols else "categorical"),
                   "imputation": "training-set median" if col in num_cols else "training-set mode",
                   "plausible_range": f"{RANGES[col][0]}-{RANGES[col][1]}" if col in RANGES else "",
                   "missing_pct": 100 * s.isna().sum() / len(df),
                   "missing_or_invalid_pct": 100 * (s.isna().sum() + n_oor + n_code) / len(df),
                   "out_of_range_n": n_oor, "missing_code_99999_n": n_code})
e3 = pd.DataFrame(e3)
info = e3.drop_duplicates("variable").set_index("variable")[["type", "imputation", "plausible_range"]]
vals = e3.pivot(index="variable", columns="cohort", values=["missing_or_invalid_pct", "out_of_range_n", "missing_code_99999_n"])
vals.columns = [f"{a}_{b}" for a, b in vals.columns]
etable3 = info.join(vals).reset_index().sort_values(f"missing_or_invalid_pct_{COHORTS_TO_RUN[0]}", ascending=False)
save_table(etable3, "eTable3_missingness_by_sex")
display(etable3.head(35).round(1))

In [ ]:
# [Cell 60] [New] eTable 5: baseline characteristics by cohort and vital status (died during follow-up / alive), and by
#       10-year status; aggregate statistics only
# Purpose: extra descriptive table (kept current with the rebuilt male cohort).
def describe_groups(df, group, buckets):
    b, n, m, _ = buckets
    rows = []
    groups = {k: g for k, g in df.groupby(group)}
    rows.append({"variable": "N", "level": "", **{f"{group}={k}": f"{len(g):,}" for k, g in groups.items()}})
    for col in [x for x in n if x not in SPECIAL_EXCLUDE]:
        r = {"variable": col, "level": "median (IQR)"}
        for k, g in groups.items():
            v = pd.to_numeric(g[col], errors="coerce")
            if col in RANGES:
                v = v.where(v.between(*RANGES[col]))
            r[f"{group}={k}"] = f"{v.median():.1f} ({v.quantile(.25):.1f}-{v.quantile(.75):.1f})"
        rows.append(r)
    for col in [x for x in b + m if x not in SPECIAL_EXCLUDE]:
        levels = sorted([x for x in df[col].dropna().unique() if x not in MULTI_NA_CODES])
        for lev in (levels if col in m else [1]):
            r = {"variable": col, "level": str(lev)}
            for k, g in groups.items():
                cnt = int((g[col] == lev).sum())
                r[f"{group}={k}"] = (f"<{SMALL_CELL}" if 0 < cnt < SMALL_CELL else f"{cnt:,}") + f" ({100 * cnt / len(g):.1f}%)"
            rows.append(r)
    return pd.DataFrame(rows)

for c in COHORTS_TO_RUN:
    d = DF[c].copy()
    d["died_during_followup"] = d[EVENT_COL].astype(int)
    y10, _, k10 = horizon_outcome(d[TIME_COL].values, d[EVENT_COL].values, PRIMARY, CensoringKM(d[TIME_COL], d[EVENT_COL]))
    d["status_10y"] = np.where(~k10, "censored_before_10y", np.where(y10 == 1, "died_by_10y", "alive_at_10y"))
    for grp in ["died_during_followup", "status_10y"]:
        tab = describe_groups(d, grp, BUCKETS[c])
        save_table(tab, f"eTable5_baseline_{c}_by_{grp}", suppress=False)
tick("SHAP and descriptive tables")

## Section 12. Internal validation of the LightGBM strategy (eTable 8)

Each resample / fold repeats the whole development procedure: variable grouping and preprocessing
(out-of-range handling, imputation, encoding, all fitted on that sample) -> fixed-horizon labels ->
LightGBM fit. Algorithm selection and recalibration are not repeated here (see section 13 for selection).

In [ ]:
# [Cell 61] [New] One complete development run (preprocessing + labels + fit + prediction)
# Purpose: R1.5, R2.2 ("which steps were repeated within each resample or fold").
def develop_predict(train_df, eval_dfs, h, names=("LightGBM",)):
    """Fit preprocessing and the named classifiers on train_df; return {name: (p_train, [p_eval, ...])}."""
    with contextlib.redirect_stdout(io.StringIO()):
        bk = build_column_buckets(train_df, BINARY_TXT, MULTI_TXT, special_exclude=SPECIAL_EXCLUDE,
                                  use_dtype_for_numeric=USE_DTYPE_FOR_NUMERIC)
    frames = {}
    if any(nm in TREE_MODELS for nm in names):
        frames[True] = preprocess_split(train_df, eval_dfs, bk, scale=False)
    if any(nm not in TREE_MODELS for nm in names):
        frames[False] = preprocess_split(train_df, eval_dfs, bk, scale=True)
    out = {}
    for nm in names:
        tr, outs, _ = frames[nm in TREE_MODELS]
        Xtr, ytr = make_horizon_dataset(tr, h, label_frame=tr)
        model = new_clf(nm, ytr).fit(Xtr, ytr)
        f = lambda d: model.predict_proba(d.drop(columns=[TIME_COL, EVENT_COL])[Xtr.columns])[:, 1]
        out[nm] = (f(tr), [f(o) for o in outs])
    return out

def perf(df, p, h, km, names=("AUROC", "Brier", "cal_intercept", "cal_slope")):
    y, w, known = horizon_outcome(df[TIME_COL].values, df[EVENT_COL].values, h, km)
    return {f"{nm}_{sch}": METRICS[nm](y, p, scheme_weights(w, known, sch)) for sch in SCHEMES for nm in names}

In [ ]:
# [Cell 62] [New] Bootstrap optimism correction (N_BOOT_OPT resamples) for AUROC, Brier score and calibration
# Purpose: R1.5, R2.2.
if RUN["internal_validation"]:
    opt_rows = []
    for c in [x for x in OPTIMISM_COHORTS if x in COHORTS_TO_RUN]:
        df = DF[c].reset_index(drop=True)
        km_full = CensoringKM(df[TIME_COL], df[EVENT_COL])
        for h in HORIZONS:
            p_app, _ = develop_predict(df, [df], h)["LightGBM"]
            apparent = perf(df, p_app, h, km_full)
            rng, opt = np.random.default_rng(SEED), []
            for b in range(N_BOOT_OPT):
                bs = df.iloc[rng.integers(0, len(df), len(df))].reset_index(drop=True)
                p_bs, (p_orig,) = develop_predict(bs, [df], h)["LightGBM"]
                a_bs = perf(bs, p_bs, h, CensoringKM(bs[TIME_COL], bs[EVENT_COL]))
                t_or = perf(df, p_orig, h, km_full)
                opt.append({k: a_bs[k] - t_or[k] for k in a_bs})
                if (b + 1) % 50 == 0: print(f"  {c} {h:.0f} y: {b + 1}/{N_BOOT_OPT}")
            opt = pd.DataFrame(opt)
            for k in apparent:
                metric, sch = k.rsplit("_", 1)
                opt_rows.append({"cohort": c, "horizon": h, "scheme": sch, "metric": metric, "apparent": apparent[k],
                                 "optimism": opt[k].mean(), "optimism_corrected": apparent[k] - opt[k].mean(),
                                 "n_boot": len(opt), "n": len(df)})
    optimism = pd.DataFrame(opt_rows)
    display(optimism.round(3))

In [ ]:
# [Cell 63] [New] Repeated stratified cross-validation (CV_FOLDS x CV_REPEATS)
# Purpose: R1.5, R2.2.
if RUN["internal_validation"]:
    cv_rows = []
    for c in [x for x in OPTIMISM_COHORTS if x in COHORTS_TO_RUN]:
        df = DF[c].reset_index(drop=True)
        for h in HORIZONS:
            folds = []
            for tr_i, te_i in RepeatedStratifiedKFold(n_splits=CV_FOLDS, n_repeats=CV_REPEATS, random_state=SEED).split(df, df[EVENT_COL]):
                tr, te = df.iloc[tr_i], df.iloc[te_i]
                _, (p_te,) = develop_predict(tr, [te], h)["LightGBM"]
                folds.append(perf(te, p_te, h, CensoringKM(tr[TIME_COL], tr[EVENT_COL])))
            folds = pd.DataFrame(folds)
            for k in folds.columns:
                metric, sch = k.rsplit("_", 1)
                cv_rows.append({"cohort": c, "horizon": h, "scheme": sch, "metric": metric,
                                "cv_mean": folds[k].mean(), "cv_sd": folds[k].std(),
                                "cv_min": folds[k].min(), "cv_max": folds[k].max(),
                                "cv": f"{CV_FOLDS}-fold x {CV_REPEATS}"})
    etable8 = optimism.merge(pd.DataFrame(cv_rows), on=["cohort", "horizon", "scheme", "metric"], how="outer")
    save_table(etable8, "eTable8_internal_validation", suppress=False)
    display(etable8[etable8.metric == "AUROC"].round(3))
    tick("internal validation")

## Section 13. Nested cross-validation of the complete development procedure, including algorithm selection

Outer loop: stratified K-fold. Inside each outer training part, the original procedure is repeated:
80/20 split into development and selection sets, preprocessing fitted on the development set, the six
classifiers and the elastic-net Cox model fitted, and the model with the highest selection-set AUROC chosen.
The chosen model is then evaluated on the untouched outer fold. This estimates the optimism caused by selecting
the best algorithm on held-out data (R1.5, R2.2) and gives a cross-validated comparison of all algorithms (R1.4).

In [ ]:
# [Cell 64] [New] Nested CV with algorithm selection
# Purpose: R1.5, R2.2, R1.4.
if RUN["nested_selection_cv"]:
    nest_rows = []
    for c in [x for x in NESTED_COHORTS if x in COHORTS_TO_RUN]:
        df = DF[c].reset_index(drop=True)
        splitter = RepeatedStratifiedKFold(n_splits=NESTED_FOLDS, n_repeats=NESTED_REPEATS, random_state=SEED)
        for fold, (tr_i, te_i) in enumerate(splitter.split(df, df[EVENT_COL])):
            outer_tr, outer_te = df.iloc[tr_i], df.iloc[te_i]
            dev, sel = train_test_split(outer_tr, test_size=TEST_SIZE, random_state=SEED, stratify=outer_tr[EVENT_COL])
            km_dev, km_out = CensoringKM(dev[TIME_COL], dev[EVENT_COL]), CensoringKM(outer_tr[TIME_COL], outer_tr[EVENT_COL])
            with contextlib.redirect_stdout(io.StringIO()):
                bk = build_column_buckets(dev, BINARY_TXT, MULTI_TXT, special_exclude=SPECIAL_EXCLUDE,
                                          use_dtype_for_numeric=USE_DTYPE_FOR_NUMERIC)
            trC, (selC, teC), _ = preprocess_split(dev, [sel, outer_te], bk, scale=True)
            cox = CoxnetSurvivalAnalysis(l1_ratio=0.5, alphas=None, max_iter=200000)
            cox.fit(trC.drop(columns=[TIME_COL, EVENT_COL]),
                    Surv.from_arrays(trC[EVENT_COL].astype(bool).values, trC[TIME_COL].astype(float).values))
            cox_sel = cox.predict(selC.drop(columns=[TIME_COL, EVENT_COL]))
            cox_te = cox.predict(teC.drop(columns=[TIME_COL, EVENT_COL]))
            for h in HORIZONS:
                fits = develop_predict(dev, [sel, outer_te], h, names=CLF_ORDER)
                sel_scores = {nm: v[1][0] for nm, v in fits.items()}
                te_scores = {nm: v[1][1] for nm, v in fits.items()}
                sel_scores["CoxNet"], te_scores["CoxNet"] = cox_sel, cox_te
                ys, ws, ks = horizon_outcome(sel[TIME_COL].values, sel[EVENT_COL].values, h, km_dev)
                yt, wt, kt = horizon_outcome(outer_te[TIME_COL].values, outer_te[EVENT_COL].values, h, km_out)
                sel_auc = {nm: w_auc(ys, s, scheme_weights(ws, ks, PRIMARY_SCHEME)) for nm, s in sel_scores.items()}
                chosen = max(sel_auc, key=sel_auc.get)
                for nm, s in te_scores.items():
                    row = {"cohort": c, "fold": fold, "horizon": h, "model": nm, "selection_AUROC": sel_auc[nm],
                           "chosen": nm == chosen}
                    for sch in SCHEMES:
                        row[f"outer_AUROC_{sch}"] = w_auc(yt, s, scheme_weights(wt, kt, sch))
                    nest_rows.append(row)
            print(f"  {c} fold {fold + 1}/{NESTED_FOLDS * NESTED_REPEATS} done")
    nested_cv = pd.DataFrame(nest_rows)
    nested_cv.to_csv(ROWLEVEL / "nested_cv_fold_level.csv", index=False)     # fold-level, aggregate but kept inside
    summ = []
    for (c, h), g in nested_cv.groupby(["cohort", "horizon"]):
        ch = g[g.chosen]
        r = {"cohort": c, "horizon": h, "n_outer_folds": len(ch),
             "selected_inner_AUROC_mean": ch["selection_AUROC"].mean(),
             "selection_optimism": (ch["selection_AUROC"] - ch[f"outer_AUROC_{PRIMARY_SCHEME}"]).mean()}
        for sch in SCHEMES:
            r[f"selection_procedure_outer_AUROC_{sch}_mean"] = ch[f"outer_AUROC_{sch}"].mean()
            r[f"selection_procedure_outer_AUROC_{sch}_sd"] = ch[f"outer_AUROC_{sch}"].std()
        r["times_selected"] = "; ".join(f"{k}: {v}" for k, v in ch["model"].value_counts().items())
        summ.append(r)
    nested_summary = pd.DataFrame(summ)
    algo = (nested_cv.groupby(["cohort", "horizon", "model"])
            .agg(**{f"outer_AUROC_{sch}_mean": (f"outer_AUROC_{sch}", "mean") for sch in SCHEMES},
                 **{f"outer_AUROC_{sch}_sd": (f"outer_AUROC_{sch}", "std") for sch in SCHEMES},
                 times_selected=("chosen", "sum")).reset_index())
    wide = nested_cv.pivot_table(index=["cohort", "horizon", "fold"], columns="model", values=f"outer_AUROC_{PRIMARY_SCHEME}")
    pdiff = []
    for m in [x for x in wide.columns if x != "LightGBM"]:
        d = (wide["LightGBM"] - wide[m]).groupby(level=[0, 1])
        pdiff.append(pd.DataFrame({"comparison": f"LightGBM - {m}", "mean_difference": d.mean(), "sd": d.std(),
                                   "folds_LightGBM_higher": d.apply(lambda x: int((x > 0).sum())),
                                   "n_folds": d.size()}).reset_index())
    save_table(nested_summary, "eTable8b_nested_cv_selection", suppress=False)
    save_table(algo, "eTable8c_nested_cv_by_algorithm", suppress=False)
    save_table(pd.concat(pdiff, ignore_index=True), "eTable8d_nested_cv_paired_differences", suppress=False)
    display(nested_summary.round(3)); display(algo.round(3))
    tick("nested CV")

## Section 14. Incremental value of predictor domains (eTable 10)

Cumulative models: demographic -> + clinical -> + functional -> + perceived health -> + sensory -> + oral
-> + social -> + dietary -> + remaining predictors (= full model). Drop-one models: full model without each block.
LightGBM and logistic regression, same training / calibration / test split as the main analysis, Platt
recalibration on the calibration set (fitted with the weights of the scheme used for evaluation). Discrimination, calibration and net benefit,
with paired bootstrap differences.

In [ ]:
# [Cell 65] [New] Domain blocks per cohort
# Purpose: R1.6, R2.7.
def blocks_for(c):
    present = set(DF[c].columns) - set(SPECIAL_EXCLUDE)
    blocks = {"demographic": [v for v in DEMOGRAPHIC_VARS if v in present],
              "clinical": [v for v in CLINICAL_VARS if v in present]}
    for d in DOMAIN_ORDER:
        blocks[d] = [v for v in DOMAIN_VARS[d] if v in present]
    used = set(sum(blocks.values(), []))
    blocks["remaining"] = sorted(present - used)
    return {k: v for k, v in blocks.items() if v}

NESTED_SETS = {}
for c in COHORTS_TO_RUN:
    blocks = blocks_for(c)
    names, sets, acc = list(blocks), {}, []
    for i, b in enumerate(names):
        acc = acc + blocks[b]
        sets[f"C{i + 1} " + ("demographic" if i == 0 else f"+ {b}")] = list(acc)
    for b in names[1:]:
        sets[f"D full - {b}"] = [v for v in acc if v not in blocks[b]]
    NESTED_SETS[c] = sets
    print(c, {k: len(v) for k, v in sets.items()})

In [ ]:
# [Cell 66] [New] Fit the domain models, recalibrate, evaluate
# Purpose: R1.6 ("assess improvements in calibration and net benefit as well as discrimination").
if RUN["domain_models"]:
    dom_rows, DOM_P = [], {}
    for c in COHORTS_TO_RUN:
        multi_cols = BUCKETS[c][2]
        for h in HORIZONS:
            y, w, known = YW[(c, h)]
            yc, wc, kc = YW_CAL[(c, h)]
            for algo in ["LightGBM", "LogReg"]:
                tr, cl, te = (TR_T[c], CL_T[c], TE_T[c]) if algo == "LightGBM" else (TR_C[c], CL_C[c], TE_C[c])
                Xtr_all, ytr = make_horizon_dataset(tr, h, label_frame=TR_T[c])
                for label, vars_ in NESTED_SETS[c].items():
                    cols = [col for col in Xtr_all.columns if parent_of(col, multi_cols) in set(vars_)]
                    est = new_clf(algo, ytr).fit(Xtr_all[cols], ytr)
                    p_raw = est.predict_proba(te[cols])[:, 1]
                    pc_raw = est.predict_proba(cl[cols])[:, 1]
                    for sch in SCHEMES:
                        ww = scheme_weights(w, known, sch)
                        p = Platt().fit(pc_raw, yc, scheme_weights(wc, kc, sch))(p_raw)
                        DOM_P[(c, h, algo, label, sch)] = p
                        row = {"cohort": c, "horizon": h, "algorithm": algo, "model": label, "scheme": sch,
                               "n_variables": len(vars_), "n_columns": len(cols)}
                        row.update(metric_row(y, p, ww, BOOT[(c, h)], ["AUROC"]))
                        row.update(metric_row(y, p, ww, None, ["Brier", "scaled_Brier", "cal_intercept", "cal_slope"]))
                        row["cal_slope_before_recalibration"] = cal_slope_w(y, p_raw, ww)
                        for t in NB_THRESHOLDS:
                            row[f"net_benefit_{int(t * 100)}pct"] = net_benefit(y, p, ww, t)
                        dom_rows.append(row)
            print(f"  {c} {h:.0f} y done")
    domain_models = pd.DataFrame(dom_rows)
    domain_models = domain_models.drop(columns=[x for x in domain_models.columns
                                                if x.endswith(("_lo", "_hi", "_fmt")) and not x.startswith("AUROC")])
    save_table(domain_models, "eTable10_domain_models", suppress=False)
    display(domain_models[(domain_models.horizon == PRIMARY) & (domain_models.scheme == PRIMARY_SCHEME)]
            [["cohort", "algorithm", "model", "n_columns", "AUROC_fmt", "Brier", "cal_slope"]])

In [ ]:
# [Cell 67] [New] Paired differences between domain models (delta AUROC, delta Brier, delta net benefit, 95% CI)
# Purpose: R1.6.
if RUN["domain_models"]:
    dd_rows = []
    for c in COHORTS_TO_RUN:
        labels = list(NESTED_SETS[c])
        cum = [l for l in labels if l.startswith("C")]
        full = cum[-1]
        pairs = [(cum[i], cum[i - 1]) for i in range(1, len(cum))] + [(full, cum[min(1, len(cum) - 1)])]
        pairs += [(full, l) for l in labels if l.startswith("D")]
        for h in HORIZONS:
            y, w, known = YW[(c, h)]
            idx = BOOT[(c, h)]
            for sch in SCHEMES:
                ww = scheme_weights(w, known, sch)
                for algo in ["LightGBM", "LogReg"]:
                    for a, b in pairs:
                        pa, pb = DOM_P[(c, h, algo, a, sch)], DOM_P[(c, h, algo, b, sch)]
                        row = {"cohort": c, "horizon": h, "scheme": sch, "algorithm": algo, "comparison": f"{a} vs {b}"}
                        fns = {"AUROC": (w_auc, idx), "Brier": (w_brier, idx[:N_BOOT_SECONDARY])}
                        fns.update({f"NB_{int(t * 100)}pct": ((lambda tt: lambda yy, pp, w_: net_benefit(yy, pp, w_, tt))(t),
                                                               idx[:N_BOOT_SECONDARY]) for t in NB_THRESHOLDS})
                        for nm, (fn, ii) in fns.items():
                            d = boot_metric(lambda yy, xa, xb, w_: fn(yy, xa, w_) - fn(yy, xb, w_), ii, y, pa, pb, ww)
                            pt = fn(y, pa, ww) - fn(y, pb, ww)
                            lo, hi = ci(d)
                            row.update({f"delta_{nm}": pt, f"delta_{nm}_lo": lo, f"delta_{nm}_hi": hi})
                        dd_rows.append(row)
    domain_diffs = pd.DataFrame(dd_rows)
    save_table(domain_diffs, "eTable10b_domain_model_differences", suppress=False)
    display(domain_diffs[(domain_diffs.horizon == PRIMARY) & (domain_diffs.scheme == PRIMARY_SCHEME)].round(4))

In [ ]:
# [Cell 68] [New] eFigure: AUROC of the cumulative domain models (10 years)
# Purpose: R1.6.
if RUN["domain_models"]:
    fig, axes = plt.subplots(1, 2, figsize=(6.7, 3.4), sharey=True, layout="constrained")
    for ax, algo in zip(axes, ["LightGBM", "LogReg"]):
        for k, c in enumerate(COHORTS_TO_RUN):
            g = domain_models[(domain_models.cohort == c) & (domain_models.horizon == PRIMARY) &
                              (domain_models.scheme == PRIMARY_SCHEME) & (domain_models.algorithm == algo) &
                              (domain_models.model.str.startswith("C"))]
            x = np.arange(len(g)) + (k - 1) * 0.12
            err = np.clip(np.vstack([g["AUROC"] - g["AUROC_lo"], g["AUROC_hi"] - g["AUROC"]]), 0, None)
            ax.errorbar(x, g["AUROC"], yerr=err, fmt="o-", ms=3, lw=0.9, capsize=1.5,
                        color=COHORT_COLOR[c], label=COHORT_LABEL[c])
        ax.set_xticks(np.arange(len(g)))
        ax.set_xticklabels([m.split(" ", 1)[1] for m in g["model"]], rotation=45, ha="right")
        ax.set_title(MODEL_LABEL[algo]); ax.grid(axis="y", lw=0.3, alpha=0.5)
    axes[0].set_ylabel("10-year AUROC (95% CI)")
    axes[0].legend(frameon=False)
    save_fig(fig, "eFigure_domain_models_10y")
    tick("domain models")

## Section 15. Pooled, interaction and sex-specific models on identical participants

Uses the both-sex training / calibration / test split only, so every model is evaluated on exactly the same
women and men. Pooled: the main both-sex models. Interaction: logistic regression with all predictor x sex
terms (LightGBM already models interactions). Sex-specific: refitted on the women (or men) of the both-sex
training set, on the predictors common to both sexes. The female-only predictors in the separate female file
cannot be included here because the two files are not linked by a participant identifier.

In [ ]:
# [Cell 69] [New] Fit and evaluate pooled / interaction / sex-specific models within each sex
# Purpose: R1.7, R2.8.
if RUN["sex_comparison"] and "bothsex" in COHORTS_TO_RUN:
    c0 = "bothsex"
    male_tr = (TRAIN[c0][SEX_COL] == MALE_CODE).values
    male_cl = (CAL[c0][SEX_COL] == MALE_CODE).values
    male_te = (TEST[c0][SEX_COL] == MALE_CODE).values
    SEXP, sex_rows = {}, []
    for h in HORIZONS:
        y, w, known = YW[(c0, h)]
        yc, wc, kc = YW_CAL[(c0, h)]
        keep_tr = horizon_outcome(TRAIN[c0][TIME_COL].values, TRAIN[c0][EVENT_COL].values, h, KM[c0])[2]
        ytr_all = horizon_outcome(TRAIN[c0][TIME_COL].values, TRAIN[c0][EVENT_COL].values, h, KM[c0])[0]
        feat = lambda d: d.drop(columns=[TIME_COL, EVENT_COL])
        # interaction design (scaled frames): every predictor plus predictor x male
        def inter(d, male):
            X = feat(d)
            Xi = X.drop(columns=[SEX_COL]).mul(male.astype(float), axis=0).add_suffix("__x_male")
            return pd.concat([X, Xi], axis=1)
        Xi_tr, Xi_cl, Xi_te = inter(TR_C[c0], male_tr), inter(CL_C[c0], male_cl), inter(TE_C[c0], male_te)
        lr_int = new_clf("LogReg", ytr_all[keep_tr]).fit(Xi_tr[keep_tr], ytr_all[keep_tr])
        preds = {"LightGBM pooled": (PRED[(c0, "LightGBM", h)], PRED_CAL[(c0, "LightGBM", h)]),
                 "LogReg pooled": (PRED[(c0, "LogReg", h)], PRED_CAL[(c0, "LogReg", h)]),
                 "LogReg pooled + sex interactions": (lr_int.predict_proba(Xi_te)[:, 1], lr_int.predict_proba(Xi_cl)[:, 1])}
        for s_lab, s_tr, s_cl, s_te in [("female", ~male_tr, ~male_cl, ~male_te), ("male", male_tr, male_cl, male_te)]:
            spec = {}
            for algo, (trf, clf_, tef) in {"LightGBM": (TR_T, CL_T, TE_T), "LogReg": (TR_C, CL_C, TE_C)}.items():
                m_ = s_tr & keep_tr
                Xs = feat(trf[c0]).drop(columns=[SEX_COL])
                est = new_clf(algo, ytr_all[m_]).fit(Xs[m_], ytr_all[m_])
                spec[f"{algo} sex-specific"] = (est.predict_proba(feat(tef[c0]).drop(columns=[SEX_COL]))[:, 1],
                                                est.predict_proba(feat(clf_[c0]).drop(columns=[SEX_COL]))[:, 1])
            allp = {**preds, **spec}
            ys, ws, ks = y[s_te], w[s_te], known[s_te]
            ycs, wcs_, kcs = yc[s_cl], wc[s_cl], kc[s_cl]
            idx = boot_idx(horizon_strata(ys, ks), N_BOOT, seed=SEED + 700 + int(h))
            for nm, (p_te, p_cl) in allp.items():
                for sch in SCHEMES:
                    ww = scheme_weights(ws, ks, sch)
                    versions = {"original": p_te[s_te],
                                "Platt (sex-specific)": Platt().fit(p_cl[s_cl], ycs, scheme_weights(wcs_, kcs, sch))(p_te[s_te])}
                    if "pooled" in nm:
                        versions["Platt (pooled)"] = Platt().fit(p_cl, yc, scheme_weights(wc, kc, sch))(p_te[s_te])
                    for v, pv in versions.items():
                        SEXP[(h, s_lab, nm, v, sch)] = pv
                        row = {"horizon": h, "sex": s_lab, "model": nm, "version": v, "scheme": sch,
                               "n_test": int(s_te.sum()), "n_known_status": int(ks.sum()), "deaths_by_horizon": int(ys[ks].sum())}
                        row.update(metric_row(ys, pv, ww, idx, ["AUROC"]))
                        row.update(metric_row(ys, pv, ww, idx[:N_BOOT_SECONDARY], ["Brier", "cal_intercept", "cal_slope", "OE_ratio"]))
                        sex_rows.append(row)
            # paired differences against the pooled LightGBM, same people
            for sch in SCHEMES:
                ww = scheme_weights(ws, ks, sch)
                for v in ["original", "Platt (sex-specific)"]:
                    base = SEXP[(h, s_lab, "LightGBM pooled", v, sch)]
                    for nm in [k for k in allp if k != "LightGBM pooled"]:
                        pv = SEXP[(h, s_lab, nm, v, sch)]
                        r = {"horizon": h, "sex": s_lab, "scheme": sch, "version": v, "comparison": f"{nm} - LightGBM pooled"}
                        for mn, fn, ii in [("AUROC", w_auc, idx), ("Brier", w_brier, idx[:N_BOOT_SECONDARY])]:
                            d = boot_metric(lambda yy, a, b, w_: fn(yy, a, w_) - fn(yy, b, w_), ii, ys, pv, base, ww)
                            pt = fn(ys, pv, ww) - fn(ys, base, ww)
                            lo, hi = ci(d)
                            r.update({f"delta_{mn}": pt, f"delta_{mn}_lo": lo, f"delta_{mn}_hi": hi, f"delta_{mn}_fmt": fmt(pt, (lo, hi), 4)})
                        sex_rows.append({"table": "difference", **r})
        print(f"  {h:.0f} y done")
    sexcmp = pd.DataFrame(sex_rows)
    save_table(sexcmp[sexcmp.get("table").isna()].drop(columns=["table"]), "eTable14_pooled_vs_sex_specific")
    save_table(sexcmp[sexcmp.get("table") == "difference"].dropna(axis=1, how="all").drop(columns=["table"]),
               "eTable14b_pooled_vs_sex_specific_differences", suppress=False)
    display(sexcmp[(sexcmp.get("table") == "difference") & (sexcmp.horizon == PRIMARY) & (sexcmp.scheme == PRIMARY_SCHEME)]
            [["sex", "version", "comparison", "delta_AUROC_fmt", "delta_Brier_fmt"]])
    tick("sex comparison")

## Section 16. Missing-data sensitivity analyses (eTable 11)

Three alternatives to the main single imputation, each refitted on the training set and recalibrated on the
calibration set (preprocessing never uses test data or outcomes):
(a) multiple imputation by chained equations for continuous predictors (M_IMPUTATIONS imputations, posterior draws;
    binary and categorical predictors after mode imputation used as auxiliary variables);
(b) no imputation: LightGBM's native handling of missing values, missing categories as their own indicator;
(c) single imputation plus missing-value indicators for predictors with >= 1% missing values.
Performance, predictor ranks and SHAP directions are compared with the main analysis.

In [ ]:
# [Cell 70] [New] Raw (unimputed) predictor frames with out-of-range values and 99999 codes set to missing
# Purpose: R1.8, R2.5.
def raw_predictors(d, c):
    b, n, m, _ = BUCKETS[c]
    b = [x for x in b if x not in SPECIAL_EXCLUDE]
    num = d[n].apply(pd.to_numeric, errors="coerce").copy()
    for col in n:
        if col in RANGES:
            lo_, hi_ = RANGES[col]
            num.loc[(num[col] < lo_) | (num[col] > hi_), col] = np.nan
    cat = d[m].replace({v: np.nan for v in MULTI_NA_CODES}).apply(pd.to_numeric, errors="coerce")
    return d[b].apply(pd.to_numeric, errors="coerce"), num, cat

def eval_alt(c, h, p_raw_te, p_raw_cl, label, extra=None):
    y, w, known = YW[(c, h)]
    yc, wc, kc = YW_CAL[(c, h)]
    rows = []
    for sch in SCHEMES:
        ww = scheme_weights(w, known, sch)
        p = Platt().fit(p_raw_cl, yc, scheme_weights(wc, kc, sch))(p_raw_te)
        r = {"cohort": c, "horizon": h, "strategy": label, "scheme": sch, **(extra or {})}
        r.update(metric_row(y, p, ww, None, ["AUROC", "Brier", "scaled_Brier", "cal_intercept", "cal_slope"]))
        r["cal_slope_before_recalibration"] = cal_slope_w(y, p_raw_te, ww)
        rows.append(r)
    return rows

def shap_summary(c, h, model, X, label, rep=1):
    """Feature- and variable-level SHAP importance with directions, for comparison with the main analysis."""
    sv = shap_matrix(model, X)
    multi_cols = BUCKETS[c][2]
    feat = pd.DataFrame({"feature": X.columns, "mean_abs_shap": np.abs(sv).mean(0),
                         "direction": [shap_direction(X[f].values, sv[:, j]) for j, f in enumerate(X.columns)]})
    feat["rank"] = feat["mean_abs_shap"].rank(ascending=False, method="min").astype(int)
    parents = [parent_of(f, multi_cols) for f in X.columns]
    grp = pd.DataFrame(sv, columns=X.columns).T.groupby(parents).sum().T
    var = pd.DataFrame({"variable": grp.columns, "grouped_mean_abs_shap": np.abs(grp.values).mean(0)})
    var["grouped_rank"] = var["grouped_mean_abs_shap"].rank(ascending=False, method="min").astype(int)
    for d_ in (feat, var):
        d_.insert(0, "replicate", rep); d_.insert(0, "strategy", label); d_.insert(0, "horizon", h); d_.insert(0, "cohort", c)
    return feat, var

MISSING_HORIZONS = [PRIMARY]
if RUN["missing_data"]:
    MISS_PERF, MISS_FEAT, MISS_VAR = [], [], []
    for c in COHORTS_TO_RUN:
        for h in MISSING_HORIZONS:
            keep_tr = horizon_outcome(TRAIN[c][TIME_COL].values, TRAIN[c][EVENT_COL].values, h, KM[c])[2]
            ytr = horizon_outcome(TRAIN[c][TIME_COL].values, TRAIN[c][EVENT_COL].values, h, KM[c])[0][keep_tr]
            f_, v_ = shap_summary(c, h, CLF_FIT[(c, "LightGBM", h)], SHAP_X[(c, h)], "main (single imputation)")
            MISS_FEAT.append(f_); MISS_VAR.append(v_)
            MISS_PERF += eval_alt(c, h, PRED[(c, "LightGBM", h)], PRED_CAL[(c, "LightGBM", h)], "main (single imputation)")
    print("main analysis entered for comparison")

In [ ]:
# [Cell 71] [New] (a) Multiple imputation by chained equations
# Purpose: R1.8, R2.5 (multiple imputation, fitted on the training set and applied to calibration / test sets).
if RUN["missing_data"]:
    for c in COHORTS_TO_RUN:
        num_cols = [x for x in BUCKETS[c][1] if x not in SPECIAL_EXCLUDE]
        aux_cols = [col for col in TR_T[c].columns if col not in num_cols + [TIME_COL, EVENT_COL]]
        parts = {}
        for part, raw, proc in [("train", TRAIN[c], TR_T[c]), ("cal", CAL[c], CL_T[c]), ("test", TEST[c], TE_T[c])]:
            parts[part] = (raw_predictors(raw, c)[1].reset_index(drop=True), proc.reset_index(drop=True))
        for h in MISSING_HORIZONS:
            keep_tr = horizon_outcome(TRAIN[c][TIME_COL].values, TRAIN[c][EVENT_COL].values, h, KM[c])[2]
            ytr = horizon_outcome(TRAIN[c][TIME_COL].values, TRAIN[c][EVENT_COL].values, h, KM[c])[0][keep_tr]
            for m in range(M_IMPUTATIONS):
                imp = IterativeImputer(max_iter=10, sample_posterior=True, random_state=SEED + m,
                                       skip_complete=True, keep_empty_features=True)
                imp.fit(pd.concat([parts["train"][0], parts["train"][1][aux_cols]], axis=1))
                X = {}
                for part in ["train", "cal", "test"]:
                    num, proc = parts[part]
                    filled = proc.copy()
                    filled[num_cols] = imp.transform(pd.concat([num, proc[aux_cols]], axis=1))[:, :len(num_cols)]
                    for col in num_cols:
                        if col in RANGES: filled[col] = filled[col].clip(*RANGES[col])
                    X[part] = filled.drop(columns=[TIME_COL, EVENT_COL])[CLF_COLS[(c, "LightGBM", h)]]
                est = new_clf("LightGBM", ytr).fit(X["train"][keep_tr], ytr)
                MISS_PERF += eval_alt(c, h, est.predict_proba(X["test"])[:, 1], est.predict_proba(X["cal"])[:, 1],
                                      "multiple imputation", {"imputation": m + 1})
                f_, v_ = shap_summary(c, h, est, X["test"], "multiple imputation", m + 1)
                MISS_FEAT.append(f_); MISS_VAR.append(v_)
                print(f"  {c} {h:.0f} y: imputation {m + 1}/{M_IMPUTATIONS}")

In [ ]:
# [Cell 72] [New] (b) No imputation (LightGBM native missing-value handling) and (c) missing-value indicators
# Purpose: R1.8, R2.5 (alternative missing-data approaches).
if RUN["missing_data"]:
    for c in COHORTS_TO_RUN:
        b_tr, n_tr, m_tr = raw_predictors(TRAIN[c], c)
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(m_tr)
        def native(d):
            b_, n_, m_ = raw_predictors(d, c)
            oh = pd.DataFrame(ohe.transform(m_), columns=ohe.get_feature_names_out(m_.columns), index=d.index)
            return pd.concat([b_, n_, oh], axis=1)
        NT = {"train": native(TRAIN[c]), "cal": native(CAL[c]), "test": native(TEST[c])}
        miss_tr = pd.concat([b_tr, n_tr, m_tr], axis=1).isna()
        ind_cols = [col for col in miss_tr.columns if miss_tr[col].mean() >= 0.01]
        def with_ind(d, proc):
            b_, n_, m_ = raw_predictors(d, c)
            ind = pd.concat([b_, n_, m_], axis=1)[ind_cols].isna().astype(float).add_prefix("miss_")
            return pd.concat([proc.drop(columns=[TIME_COL, EVENT_COL]), ind], axis=1)
        MI_ = {"train": with_ind(TRAIN[c], TR_T[c]), "cal": with_ind(CAL[c], CL_T[c]), "test": with_ind(TEST[c], TE_T[c])}
        print(f"  {c}: native design {NT['train'].shape[1]} columns; {len(ind_cols)} missing indicators")
        for h in MISSING_HORIZONS:
            keep_tr = horizon_outcome(TRAIN[c][TIME_COL].values, TRAIN[c][EVENT_COL].values, h, KM[c])[2]
            ytr = horizon_outcome(TRAIN[c][TIME_COL].values, TRAIN[c][EVENT_COL].values, h, KM[c])[0][keep_tr]
            for label, X in [("no imputation (LightGBM native)", NT), ("missing indicators", MI_)]:
                est = new_clf("LightGBM", ytr).fit(X["train"][keep_tr], ytr)
                MISS_PERF += eval_alt(c, h, est.predict_proba(X["test"])[:, 1], est.predict_proba(X["cal"])[:, 1], label)
                f_, v_ = shap_summary(c, h, est, X["test"], label)
                MISS_FEAT.append(f_); MISS_VAR.append(v_)

In [ ]:
# [Cell 73] [New] eTable 11: summary of the missing-data sensitivity analyses
# Purpose: R1.8 ("whether the principal rankings and directions persist"), R2.5.
if RUN["missing_data"]:
    mp = pd.DataFrame(MISS_PERF)
    perf_cols = ["AUROC", "Brier", "scaled_Brier", "cal_intercept", "cal_slope", "cal_slope_before_recalibration"]
    mi_perf = (mp.groupby(["cohort", "horizon", "strategy", "scheme"])[perf_cols]
               .agg(["mean", "min", "max"]).reset_index())
    mi_perf.columns = ["_".join([x for x in col if x]) for col in mi_perf.columns]
    save_table(mi_perf, "eTable11_missing_data_performance", suppress=False)
    mf, mv = pd.concat(MISS_FEAT, ignore_index=True), pd.concat(MISS_VAR, ignore_index=True)
    rank_rows = []
    for (c, h), g in mf.groupby(["cohort", "horizon"]):
        main_f = g[g.strategy.str.startswith("main")].set_index("feature")
        top20 = set(main_f.sort_values("rank").index[:20])
        for strat, gs in g.groupby("strategy"):
            agg = gs.groupby("feature").agg(rank_median=("rank", "median"), rank_min=("rank", "min"),
                                            rank_max=("rank", "max"),
                                            direction_positive_share=("direction", lambda x: float(np.mean(x > 0))))
            top_s = set(agg.sort_values("rank_median").index[:20])
            for f in [k for k in KEY_PREDICTORS if k in main_f.index] + sorted(top20 - set(KEY_PREDICTORS)):
                rank_rows.append({"cohort": c, "horizon": h, "strategy": strat, "feature": f, "label": LABELS.get(f, f),
                                  "rank_main": int(main_f.loc[f, "rank"]),
                                  "direction_main": direction_label(main_f.loc[f, "direction"]),
                                  "rank_median": agg["rank_median"].get(f, np.nan), "rank_min": agg["rank_min"].get(f, np.nan),
                                  "rank_max": agg["rank_max"].get(f, np.nan),
                                  "share_same_direction": (agg["direction_positive_share"].get(f, np.nan)
                                                           if main_f.loc[f, "direction"] > 0 else
                                                           1 - agg["direction_positive_share"].get(f, np.nan)),
                                  "top20_overlap_with_main": len(top20 & top_s)})
    mi_ranks = pd.DataFrame(rank_rows)
    save_table(mi_ranks, "eTable11b_missing_data_predictor_ranks", suppress=False)
    var_rows = []
    for (c, h), g in mv.groupby(["cohort", "horizon"]):
        main_v = g[g.strategy.str.startswith("main")].set_index("variable")["grouped_mean_abs_shap"]
        for strat, gs in g.groupby("strategy"):
            imp_ = gs.groupby("variable")["grouped_mean_abs_shap"].mean().reindex(main_v.index)
            rho = stats.spearmanr(main_v.values, imp_.values, nan_policy="omit")[0]
            for v in main_v.sort_values(ascending=False).index[:20]:
                var_rows.append({"cohort": c, "horizon": h, "strategy": strat, "variable": v,
                                 "grouped_rank_main": int(main_v.rank(ascending=False, method="min")[v]),
                                 "grouped_rank_strategy": int(imp_.rank(ascending=False, method="min")[v]) if np.isfinite(imp_[v]) else np.nan,
                                 "spearman_all_variables": rho})
    save_table(pd.DataFrame(var_rows), "eTable11c_missing_data_grouped_ranks", suppress=False)
    display(mi_perf[mi_perf.scheme == PRIMARY_SCHEME].round(3))
    display(mi_ranks[mi_ranks.strategy != "main (single imputation)"].head(30))
    tick("missing data")

## Section 17. Censoring and class-weighting variants for classifier training

Main analysis: participants censored before the horizon excluded, class weighting on (LightGBM `is_unbalance`,
logistic regression `class_weight="balanced"`). Variants show how these choices affect discrimination and the
absolute level of predicted risk before and after recalibration (R1.2, R2.1).

In [ ]:
# [Cell 74] [New] Training variants: class weighting off, IPCW training weights, IPCW x class balance, censored-as-alive
# Purpose: R1.2 ("how censoring weights interacted with class-balancing weights"), R2.1 (implications of
#          class weighting for probability estimation and recalibration).
def balanced_weights(y, base):
    W, W1 = base.sum(), base[y == 1].sum()
    return base * np.where(y == 1, W / (2 * W1), W / (2 * (W - W1)))

if RUN["weighting_sensitivity"]:
    wt_rows = []
    for c in COHORTS_TO_RUN:
        for h in HORIZONS:
            ytr, wtr, ktr = horizon_outcome(TRAIN[c][TIME_COL].values, TRAIN[c][EVENT_COL].values, h, KM[c])
            y, w, known = YW[(c, h)]
            yc, wc, kc = YW_CAL[(c, h)]
            for algo in ["LightGBM", "LogReg"]:
                tr, cl, te = (TR_T[c], CL_T[c], TE_T[c]) if algo == "LightGBM" else (TR_C[c], CL_C[c], TE_C[c])
                Xtr, Xcl, Xte = [d.drop(columns=[TIME_COL, EVENT_COL]) for d in (tr, cl, te)]
                def unweighted():
                    est = new_clf(algo, ytr[ktr])
                    return est.set_params(is_unbalance=False) if algo == "LightGBM" else est.set_params(class_weight=None)
                variants = {
                    "main: exclude censored, class-weighted": (new_clf(algo, ytr[ktr]), ktr, ytr, None),
                    "exclude censored, no class weights": (unweighted(), ktr, ytr, None),
                    "IPCW weights, no class weights": (unweighted(), ktr, ytr, wtr[ktr]),
                    "IPCW x class-balancing weights": (unweighted(), ktr, ytr, balanced_weights(ytr[ktr], wtr[ktr])),
                    "censored counted as alive, class-weighted": (new_clf(algo, ytr), np.ones(len(ytr), bool), ytr, None),
                }
                for label, (est, rows_, yy, sw) in variants.items():
                    est.fit(Xtr[rows_], yy[rows_], sample_weight=sw)
                    p_raw, pc_raw = est.predict_proba(Xte)[:, 1], est.predict_proba(Xcl)[:, 1]
                    for sch in SCHEMES:
                        ww, wcs = scheme_weights(w, known, sch), scheme_weights(wc, kc, sch)
                        p_pl = Platt().fit(pc_raw, yc, wcs)(p_raw)
                        for v, p in [("original", p_raw), ("Platt", p_pl)]:
                            r = {"cohort": c, "horizon": h, "algorithm": algo, "training": label, "scheme": sch, "version": v,
                                 "mean_predicted": float(np.average(p[ww > 0], weights=ww[ww > 0])),
                                 "observed_risk": w_prevalence(y, ww)}
                            r.update(metric_row(y, p, ww, None, ["AUROC", "Brier", "cal_intercept", "cal_slope", "OE_ratio"]))
                            wt_rows.append({k: v_ for k, v_ in r.items() if not k.endswith(("_lo", "_hi", "_fmt"))})
            print(f"  {c} done")
    weighting = pd.DataFrame(wt_rows)
    save_table(weighting, "eTable15_training_weight_variants", suppress=False)
    display(weighting[(weighting.horizon == PRIMARY) & (weighting.scheme == PRIMARY_SCHEME)].round(3))
    tick("weighting variants")

## Section 18. Hyperparameter tuning sensitivity

The main models use fixed, pre-specified settings (no data-driven tuning). Here LightGBM and logistic regression
are tuned on the training set only (stratified K-fold, AUROC), then evaluated once on the test set.

In [ ]:
# [Cell 75] [New] Cross-validated tuning versus fixed settings
# Purpose: R2.2 (tuning criterion, folds and search settings; distinguished from internal validation).
LGB_SPACE = {"n_estimators": [200, 400, 600, 900, 1200], "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1],
             "num_leaves": [7, 15, 31, 63], "max_depth": [-1, 3, 4, 5, 6], "min_child_samples": [10, 20, 50, 100],
             "subsample": [0.6, 0.8, 1.0], "subsample_freq": [0, 1], "colsample_bytree": [0.5, 0.7, 0.8, 1.0],
             "reg_lambda": [0.0, 1.0, 5.0, 10.0], "is_unbalance": [True, False]}
LR_SPACE = {"C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0], "penalty": ["l1", "l2"], "class_weight": [None, "balanced"]}
TUNE_HORIZONS = [PRIMARY]
if RUN["tuning_sensitivity"]:
    tune_rows = []
    for c in COHORTS_TO_RUN:
        for h in TUNE_HORIZONS:
            y, w, known = YW[(c, h)]
            yc, wc, kc = YW_CAL[(c, h)]
            for algo in ["LightGBM", "LogReg"]:
                tr, cl, te = (TR_T[c], CL_T[c], TE_T[c]) if algo == "LightGBM" else (TR_C[c], CL_C[c], TE_C[c])
                Xtr, ytr = make_horizon_dataset(tr, h, label_frame=TR_T[c])
                cv = StratifiedKFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=SEED)
                base = new_clf(algo, ytr)
                if algo == "LightGBM":
                    search = RandomizedSearchCV(base, LGB_SPACE, n_iter=TUNE_N_ITER, scoring="roc_auc", cv=cv,
                                                random_state=SEED, n_jobs=1, refit=True)
                else:
                    search = GridSearchCV(base, LR_SPACE, scoring="roc_auc", cv=cv, n_jobs=1, refit=True)
                search.fit(Xtr, ytr)
                Xte, Xcl = te[Xtr.columns], cl[Xtr.columns]
                cand = {"fixed (main)": (PRED[(c, algo, h)], PRED_CAL[(c, algo, h)]),
                        "tuned": (search.predict_proba(Xte)[:, 1], search.predict_proba(Xcl)[:, 1])}
                boots = {}
                for label, (p_raw, pc_raw) in cand.items():
                    for sch in SCHEMES:
                        ww, wcs = scheme_weights(w, known, sch), scheme_weights(wc, kc, sch)
                        p_pl = Platt().fit(pc_raw, yc, wcs)(p_raw)
                        r = {"cohort": c, "horizon": h, "algorithm": algo, "settings": label, "scheme": sch,
                             "cv_AUROC": search.best_score_ if label == "tuned" else np.nan,
                             "best_params": json.dumps(search.best_params_, default=str) if label == "tuned" else "eTable 4"}
                        r.update(metric_row(y, p_raw, ww, BOOT[(c, h)], ["AUROC"]))
                        r.update({f"original_{k}": v for k, v in metric_row(y, p_raw, ww, None, ["Brier", "cal_intercept", "cal_slope"]).items()
                                  if not k.endswith(("_lo", "_hi", "_fmt"))})
                        r.update({f"Platt_{k}": v for k, v in metric_row(y, p_pl, ww, None, ["Brier", "cal_intercept", "cal_slope"]).items()
                                  if not k.endswith(("_lo", "_hi", "_fmt"))})
                        boots[(label, sch)] = boot_metric(w_auc, BOOT[(c, h)], y, p_raw, ww)
                        tune_rows.append(r)
                for sch in SCHEMES:
                    d = boots[("tuned", sch)] - boots[("fixed (main)", sch)]
                    pt = w_auc(y, cand["tuned"][0], scheme_weights(w, known, sch)) - w_auc(y, cand["fixed (main)"][0], scheme_weights(w, known, sch))
                    tune_rows.append({"cohort": c, "horizon": h, "algorithm": algo, "settings": "tuned - fixed", "scheme": sch,
                                      "AUROC": pt, "AUROC_lo": ci(d)[0], "AUROC_hi": ci(d)[1], "AUROC_fmt": fmt(pt, ci(d))})
                print(f"  {c} {algo}: best CV AUROC {search.best_score_:.3f}")
    tuning = pd.DataFrame(tune_rows)
    save_table(tuning, "eTable4c_tuning_sensitivity", suppress=False)
    display(tuning[tuning.scheme == PRIMARY_SCHEME][["cohort", "algorithm", "settings", "AUROC_fmt", "best_params"]])
    tick("tuning")

## Section 19. Model comparison over repeated random splits

The whole split -> preprocessing -> fitting sequence is repeated with N_SPLIT_SEEDS different random seeds,
showing how much the AUROC ranking of the leading models depends on one particular split (R1.4).

In [ ]:
# [Cell 76] [New] Repeated train / calibration / test splits
# Purpose: R1.4, R2.7.
if RUN["repeated_splits"]:
    rs_rows = []
    for c in [x for x in SPLIT_COHORTS if x in COHORTS_TO_RUN]:
        df = DF[c]
        for k in range(N_SPLIT_SEEDS):
            seed = SEED + 1000 + k
            tr_df, te_df = train_test_split(df, test_size=TEST_SIZE, random_state=seed, stratify=df[EVENT_COL])
            tr_df, _ = train_test_split(tr_df, test_size=0.20, random_state=seed, stratify=tr_df[EVENT_COL])
            km = CensoringKM(tr_df[TIME_COL], tr_df[EVENT_COL])
            scores = {h: {} for h in HORIZONS}
            for h in HORIZONS:
                fits = develop_predict(tr_df, [te_df], h, names=[m for m in REPEATED_SPLIT_MODELS if m in CLF_ORDER])
                scores[h].update({nm: v[1][0] for nm, v in fits.items()})
            if "CoxNet" in REPEATED_SPLIT_MODELS:
                trC, (teC,), _ = preprocess_split(tr_df, [te_df], BUCKETS[c], scale=True)
                cox = CoxnetSurvivalAnalysis(l1_ratio=0.5, alphas=None, max_iter=200000).fit(
                    trC.drop(columns=[TIME_COL, EVENT_COL]),
                    Surv.from_arrays(trC[EVENT_COL].astype(bool).values, trC[TIME_COL].astype(float).values))
                for h in HORIZONS:
                    scores[h]["CoxNet"] = cox.predict(teC.drop(columns=[TIME_COL, EVENT_COL]))
            for h in HORIZONS:
                yt, wt, kt = horizon_outcome(te_df[TIME_COL].values, te_df[EVENT_COL].values, h, km)
                for nm, s in scores[h].items():
                    rs_rows.append({"cohort": c, "split": k + 1, "seed": seed, "horizon": h, "model": nm,
                                    **{f"AUROC_{sch}": w_auc(yt, s, scheme_weights(wt, kt, sch)) for sch in SCHEMES}})
            print(f"  {c}: split {k + 1}/{N_SPLIT_SEEDS}")
    rsplit = pd.DataFrame(rs_rows)
    rs_sum = (rsplit.groupby(["cohort", "horizon", "model"])
              .agg(**{f"AUROC_{sch}_{f}": (f"AUROC_{sch}", f) for sch in SCHEMES for f in ["mean", "std", "min", "max"]})
              .reset_index())
    rd = []
    for sch in SCHEMES:
        wide = rsplit.pivot_table(index=["cohort", "horizon", "split"], columns="model", values=f"AUROC_{sch}")
        best = wide.idxmax(axis=1)
        for m in [x for x in wide.columns if x != "LightGBM"]:
            d = (wide["LightGBM"] - wide[m]).groupby(level=[0, 1])
            rd.append(pd.DataFrame({"scheme": sch, "comparison": f"LightGBM - {m}", "mean": d.mean(),
                                    "p2.5": d.quantile(0.025), "p97.5": d.quantile(0.975),
                                    "splits_LightGBM_higher": d.apply(lambda x: int((x > 0).sum())), "n_splits": d.size()}).reset_index())
        rs_sum[f"times_best_{sch}"] = [int(((best.xs((c, h), level=[0, 1])) == m).sum())
                                       for c, h, m in zip(rs_sum.cohort, rs_sum.horizon, rs_sum.model)]
    save_table(rs_sum, "eTable6h_repeated_splits_auroc", suppress=False)
    save_table(pd.concat(rd, ignore_index=True), "eTable6i_repeated_splits_differences", suppress=False)
    display(rs_sum.round(3))
    tick("repeated splits")

## Section 20. Stability of predictor rankings, sex differences in rankings, permutation importance

LightGBM (10 years) is refitted on N_SHAP_BOOT bootstrap samples of the training set; SHAP is computed on a fixed
subsample of the test set. Rank intervals show how stable each variable's position is. Female-minus-male rank
differences are computed among the variables common to both sexes; an interval excluding 0 indicates a ranking
difference larger than refitting variability (still not evidence of effect modification, R1.7 / R2.8).
Permutation importance (drop in test AUROC when a whole variable is permuted) is a model-agnostic check.

In [ ]:
# [Cell 77] [New] Bootstrap SHAP rank stability (variable level, categorical indicators grouped)
# Purpose: R2.8, R1.7.
if RUN["shap_stability"]:
    STAB = {}
    for c in COHORTS_TO_RUN:
        multi_cols = BUCKETS[c][2]
        Xtr, ytr = make_horizon_dataset(TR_T[c], PRIMARY)
        Xte = SHAP_X[(c, PRIMARY)]
        sub = np.random.default_rng(SEED).choice(len(Xte), min(2000, len(Xte)), replace=False)
        parents = [parent_of(col, multi_cols) for col in Xte.columns]
        rng, imps = np.random.default_rng(SEED + 1), []
        for b in range(N_SHAP_BOOT):
            ii = rng.integers(0, len(Xtr), len(Xtr))
            est = new_clf("LightGBM", ytr[ii]).fit(Xtr.iloc[ii], ytr[ii])
            sv = shap_matrix(est, Xte.iloc[sub])
            grp = pd.DataFrame(sv, columns=Xte.columns).T.groupby(parents).sum().T
            imps.append(pd.Series(np.abs(grp.values).mean(0), index=grp.columns))
        STAB[c] = pd.DataFrame(imps)                                   # rows = bootstrap refits
        print(f"  {c}: {N_SHAP_BOOT} refits")
    stab_rows = []
    for c, imp in STAB.items():
        ranks = imp.rank(axis=1, ascending=False, method="min")
        main_rank = grouped_shap[(grouped_shap.cohort == c) & (grouped_shap.horizon == PRIMARY)].set_index("variable")["grouped_rank"]
        for v in main_rank.sort_values().index[:30]:
            stab_rows.append({"cohort": c, "variable": v, "grouped_rank_main": int(main_rank[v]),
                              "rank_median": ranks[v].median(), "rank_p2.5": ranks[v].quantile(0.025),
                              "rank_p97.5": ranks[v].quantile(0.975), "share_in_top10": float((ranks[v] <= 10).mean())})
    save_table(pd.DataFrame(stab_rows), "eTable12b_shap_rank_stability", suppress=False)
    if "female" in STAB and "male" in STAB:
        common = sorted(set(STAB["female"].columns) & set(STAB["male"].columns))
        rf = STAB["female"][common].rank(axis=1, ascending=False, method="min")
        rm = STAB["male"][common].rank(axis=1, ascending=False, method="min")
        d = rf.values - rm.values
        sexrank = pd.DataFrame({"variable": common, "rank_female_median": rf.median().values,
                                "rank_male_median": rm.median().values,
                                "diff_median": np.median(d, 0), "diff_p2.5": np.percentile(d, 2.5, 0),
                                "diff_p97.5": np.percentile(d, 97.5, 0)})
        sexrank["interval_excludes_0"] = (sexrank["diff_p2.5"] > 0) | (sexrank["diff_p97.5"] < 0)
        sexrank = sexrank.sort_values(["rank_female_median"]).reset_index(drop=True)
        save_table(sexrank, "eTable12c_female_vs_male_rank_differences", suppress=False)
        display(sexrank.head(25))

In [ ]:
# [Cell 78] [New] Permutation importance by whole variable (drop in 10-year test AUROC), compared with grouped SHAP
# Purpose: R2.8 (whole-variable importance, independent of SHAP).
if RUN["shap_stability"]:
    perm_rows = []
    for c in COHORTS_TO_RUN:
        multi_cols = BUCKETS[c][2]
        model, X = CLF_FIT[(c, "LightGBM", PRIMARY)], SHAP_X[(c, PRIMARY)]
        y, w, known = YW[(c, PRIMARY)]
        base_p = model.predict_proba(X)[:, 1]
        base = {sch: w_auc(y, base_p, scheme_weights(w, known, sch)) for sch in SCHEMES}
        groups = pd.Series(X.columns).groupby([parent_of(col, multi_cols) for col in X.columns]).apply(list)
        rng = np.random.default_rng(SEED + 2)
        for v, cols_ in groups.items():
            drops = {sch: [] for sch in SCHEMES}
            for r in range(PERM_REPEATS):
                Xp = X.copy()
                Xp[cols_] = X[cols_].values[rng.permutation(len(X))]
                pp = model.predict_proba(Xp)[:, 1]
                for sch in SCHEMES:
                    drops[sch].append(base[sch] - w_auc(y, pp, scheme_weights(w, known, sch)))
            perm_rows.append({"cohort": c, "variable": v, **{f"AUROC_drop_{sch}_mean": np.mean(drops[sch]) for sch in SCHEMES},
                              **{f"AUROC_drop_{sch}_sd": np.std(drops[sch]) for sch in SCHEMES}})
    perm = pd.DataFrame(perm_rows)
    perm["perm_rank"] = perm.groupby("cohort")[f"AUROC_drop_{PRIMARY_SCHEME}_mean"].rank(ascending=False, method="min")
    gs = grouped_shap[grouped_shap.horizon == PRIMARY][["cohort", "variable", "grouped_rank"]]
    perm = perm.merge(gs, on=["cohort", "variable"], how="left").sort_values(["cohort", "perm_rank"])
    for c in COHORTS_TO_RUN:
        g = perm[perm.cohort == c]
        print(f"  {c}: Spearman(permutation rank, grouped SHAP rank) = {stats.spearmanr(g.perm_rank, g.grouped_rank)[0]:.3f}")
    save_table(perm, "eTable12d_permutation_importance", suppress=False)
    tick("ranking stability")

## Section 21. Subgroup performance and an age >= 65 refit

In [ ]:
# [Cell 79] [New] Performance by age group (all cohorts) and by sex (both-sex cohort)
# Purpose: Minor 1 (middle-aged and older adults), R1.7 (pooled model within each sex).
if RUN["subgroups"]:
    sg_rows = []
    for c in COHORTS_TO_RUN:
        age = pd.to_numeric(TEST[c][AGE_COL], errors="coerce").values
        groups = {f"age {lab}": (pd.cut(age, SUBGROUP_AGE, right=False, labels=SUBGROUP_AGE_LABELS) == lab)
                  for lab in SUBGROUP_AGE_LABELS}
        if SEX_COL in TEST[c].columns:
            groups.update({"female": (TEST[c][SEX_COL] == FEMALE_CODE).values, "male": (TEST[c][SEX_COL] == MALE_CODE).values})
        for h in HORIZONS:
            y, w, known = YW[(c, h)]
            models = {"LightGBM original": PRED[(c, "LightGBM", h)],
                      "LightGBM Platt": None,                    # scheme-specific, filled below
                      "LogReg original": PRED[(c, "LogReg", h)]}
            if (c, "CoxNet", h) in SPROB:
                models["CoxNet probability"] = SPROB[(c, "CoxNet", h)]
            for gname, gm in groups.items():
                gm = np.asarray(gm, bool)
                if (y[gm & known] == 1).sum() < 10 or (y[gm & known] == 0).sum() < 10:
                    continue
                idx = boot_idx(horizon_strata(y[gm], known[gm]), N_BOOT_SECONDARY, seed=SEED + 900)
                for mname, p_ in models.items():
                    for sch in SCHEMES:
                        p = VERSIONS[(c, h, sch)]["Platt"] if p_ is None else p_
                        ww = scheme_weights(w[gm], known[gm], sch)
                        r = {"cohort": c, "horizon": h, "subgroup": gname, "model": mname, "scheme": sch,
                             "n_test": int(gm.sum()), "n_known_status": int(known[gm].sum()),
                             "deaths_by_horizon": int(y[gm & known].sum())}
                        r.update(metric_row(y[gm], p[gm], ww, idx, ["AUROC"]))
                        r.update({k: v for k, v in metric_row(y[gm], p[gm], ww, None,
                                  ["Brier", "cal_intercept", "cal_slope", "OE_ratio"]).items()
                                  if not k.endswith(("_lo", "_hi", "_fmt"))})
                        sg_rows.append(r)
    subgroups = pd.DataFrame(sg_rows)
    save_table(subgroups, "eTable16_subgroup_performance")
    display(subgroups[(subgroups.horizon == PRIMARY) & (subgroups.scheme == PRIMARY_SCHEME) &
                      (subgroups.model == "LightGBM Platt")][["cohort", "subgroup", "n_test", "AUROC_fmt", "cal_slope", "OE_ratio"]])

In [ ]:
# [Cell 80] [New] LightGBM refitted in participants aged >= 65 versus the main model, evaluated in test participants >= 65
# Purpose: Minor 1 (whether results hold in older adults only).
if RUN["subgroups"]:
    old_rows = []
    for c in COHORTS_TO_RUN:
        a_tr = pd.to_numeric(TRAIN[c][AGE_COL], errors="coerce").values >= 65
        a_cl = pd.to_numeric(CAL[c][AGE_COL], errors="coerce").values >= 65
        a_te = pd.to_numeric(TEST[c][AGE_COL], errors="coerce").values >= 65
        for h in HORIZONS:
            ytr, _, ktr = horizon_outcome(TRAIN[c][TIME_COL].values, TRAIN[c][EVENT_COL].values, h, KM[c])
            y, w, known = YW[(c, h)]
            yc, wc, kc = YW_CAL[(c, h)]
            X = lambda d: d.drop(columns=[TIME_COL, EVENT_COL])[CLF_COLS[(c, "LightGBM", h)]]
            m_ = a_tr & ktr
            est = new_clf("LightGBM", ytr[m_]).fit(X(TR_T[c])[m_], ytr[m_])
            cand = {"main model (all ages)": (PRED[(c, "LightGBM", h)][a_te], PRED_CAL[(c, "LightGBM", h)][a_cl]),
                    "refit in age >= 65": (est.predict_proba(X(TE_T[c]))[:, 1][a_te], est.predict_proba(X(CL_T[c]))[:, 1][a_cl])}
            idx = boot_idx(horizon_strata(y[a_te], known[a_te]), N_BOOT_SECONDARY, seed=SEED + 950)
            bt = {}
            for lab, (p_te, p_cl) in cand.items():
                for sch in SCHEMES:
                    ww = scheme_weights(w[a_te], known[a_te], sch)
                    p_pl = Platt().fit(p_cl, yc[a_cl], scheme_weights(wc[a_cl], kc[a_cl], sch))(p_te)
                    r = {"cohort": c, "horizon": h, "model": lab, "scheme": sch, "n_train": int(m_.sum()),
                         "n_test_65plus": int(a_te.sum()), "deaths_by_horizon": int(y[a_te & known].sum())}
                    r.update(metric_row(y[a_te], p_te, ww, idx, ["AUROC"]))
                    r.update({f"Platt_{k}": v for k, v in metric_row(y[a_te], p_pl, ww, None, ["Brier", "cal_intercept", "cal_slope"]).items()
                              if not k.endswith(("_lo", "_hi", "_fmt"))})
                    bt[(lab, sch)] = boot_metric(w_auc, idx, y[a_te], p_te, ww)
                    old_rows.append(r)
            for sch in SCHEMES:
                ww = scheme_weights(w[a_te], known[a_te], sch)
                pt = w_auc(y[a_te], cand["refit in age >= 65"][0], ww) - w_auc(y[a_te], cand["main model (all ages)"][0], ww)
                lo, hi = ci(bt[("refit in age >= 65", sch)] - bt[("main model (all ages)", sch)])
                old_rows.append({"cohort": c, "horizon": h, "model": "refit - main", "scheme": sch,
                                 "AUROC": pt, "AUROC_lo": lo, "AUROC_hi": hi, "AUROC_fmt": fmt(pt, (lo, hi))})
    older = pd.DataFrame(old_rows)
    save_table(older, "eTable16b_age65_refit")
    display(older[older.scheme == PRIMARY_SCHEME][["cohort", "horizon", "model", "AUROC_fmt"]])
    tick("subgroups")

## Section 22. Ridge Cox hazard ratios (eFigure 6) and sex-interaction tests

In [ ]:
# [Cell 81] [Original, modified] Cox HR from section 9.3 of CVD_SURE_pipeline.ipynb (CoxPHFitter(penalizer=0.1), robust=True)
# Change: all three cohorts use their complete data (the male cohort is rebuilt from the both-sex file); columns with
#         zero variance removed. Used only to describe hazard ratios, not in the performance comparison (R2.2).
if RUN["cox_hr"]:
    hr_list, FULL_PROC = [], {}
    for c in COHORTS_TO_RUN:
        df = DF[c]
        b, n, m, _ = BUCKETS[c]
        full, _, _ = preprocess_for_model(df, df, binary_cols=b, num_cols=n, multi_cat_cols=m,
                                          passthrough_cols=["edate", "survival"], scale_numeric=False, ranges=RANGES)
        full = full.loc[:, (full.std() > 0) | full.columns.isin([TIME_COL, EVENT_COL])].astype(float)
        FULL_PROC[c] = full
        cph = CoxPHFitter(penalizer=0.1).fit(full, duration_col=TIME_COL, event_col=EVENT_COL, robust=True)
        s = cph.summary.reset_index().rename(columns={"covariate": "feature"})
        s = s[["feature", "exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
        s.columns = ["feature", "HR", "HR_lo", "HR_hi", "p"]
        s.insert(0, "cohort", c)
        s["label"] = s["feature"].map(lambda f: LABELS.get(f, f))
        hr_list.append(s)
        print(f" {c}: Cox HR fitted")
    hr = pd.concat(hr_list, ignore_index=True)
    save_table(hr, "eFigure6_cox_hazard_ratios", suppress=False)

In [ ]:
# [Cell 82] [Original, modified] eFigure 6 forest plot (20 significant predictors with the largest |log HR|)
# Change: three cohorts in one figure at 170 mm width; x-axis ticks as plain numbers (was 8x10^-1 style).
if RUN["cox_hr"]:
    top = hr[hr.p < 0.05].assign(abs_log=lambda x: np.abs(np.log(x.HR)))
    feats = top.groupby("feature")["abs_log"].max().sort_values(ascending=False).head(20).index.tolist()
    fig, ax = plt.subplots(figsize=(6.7, 6.0), layout="constrained")
    for k, c in enumerate(COHORTS_TO_RUN):
        sub = hr[hr.cohort == c].set_index("feature").reindex(feats)
        ypos = np.arange(len(feats)) + (k - 1) * 0.22
        err = np.clip(np.vstack([sub["HR"] - sub["HR_lo"], sub["HR_hi"] - sub["HR"]]), 0, None)
        ax.errorbar(sub["HR"], ypos, xerr=err, fmt="o", ms=3, lw=0.8, capsize=1.5,
                    color=COHORT_COLOR[c], label=COHORT_LABEL[c])
    ax.axvline(1, color="grey", lw=0.8, ls="--")
    ax.set_xscale("log")
    # [R1 axis fix] plain-number ticks on the log axis, no minor ticks (display only)
    from matplotlib.ticker import FixedLocator, FormatStrFormatter, NullLocator
    lo, hi = ax.get_xlim()
    ax.xaxis.set_major_locator(FixedLocator([t for t in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.4, 1.6, 2.0, 2.5, 3.0)
                                             if lo <= t <= hi]))
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_yticks(np.arange(len(feats)))
    ax.set_yticklabels([LABELS.get(f, f) for f in feats])
    ax.invert_yaxis()
    ax.set_xlabel("Hazard ratio (95% CI, robust SE)")
    ax.legend(frameon=False)
    save_fig(fig, "eFigure6_cox_hazard_ratios")

In [ ]:
# [Cell 83] [New] Sex x predictor interaction tests in Cox models (both-sex cohort, complete data)
# Purpose: R1.7 ("SHAP rankings ... do not demonstrate effect modification"): formal tests of effect modification
#          for the leading predictors. Each model: age + sex + predictor (+ predictor x sex); likelihood-ratio test.
#          Continuous predictors per SD; categorical predictors with the most common category as reference.
if RUN["cox_hr"] and "bothsex" in COHORTS_TO_RUN:
    full = FULL_PROC["bothsex"] if "bothsex" in FULL_PROC else None
    if full is None:
        b, n, m, _ = BUCKETS["bothsex"]
        full, _, _ = preprocess_for_model(DF["bothsex"], DF["bothsex"], binary_cols=b, num_cols=n, multi_cat_cols=m,
                                          passthrough_cols=["edate", "survival"], scale_numeric=False, ranges=RANGES)
    multi_cols = BUCKETS["bothsex"][2]
    gsh = grouped_shap[grouped_shap.horizon == PRIMARY]
    cand = list(gsh[gsh.cohort == "bothsex"].sort_values("grouped_rank")["variable"].head(15))
    for c in ["female", "male"]:
        if c in COHORTS_TO_RUN:
            cand += list(gsh[gsh.cohort == c].sort_values("grouped_rank")["variable"].head(10))
    cand = [v for v in dict.fromkeys(cand) if v not in (SEX_COL, AGE_COL)] + [AGE_COL]
    male = (DF["bothsex"][SEX_COL].values == MALE_CODE).astype(float)
    age_sd = (full[AGE_COL] - full[AGE_COL].mean()) / full[AGE_COL].std()
    int_rows = []
    for v in cand:
        cols_ = [col for col in full.columns if parent_of(col, multi_cols) == v]
        if not cols_:
            continue
        X = full[cols_].astype(float).copy()
        if v in multi_cols:
            ref = X.sum().idxmax()
            X = X.drop(columns=[ref])
        elif X.shape[1] == 1 and X.iloc[:, 0].nunique() > 2:
            X = (X - X.mean()) / X.std()
        X = X.loc[:, X.std() > 0]
        if X.shape[1] == 0:
            continue
        base = pd.DataFrame({"T": full[TIME_COL].values, "E": full[EVENT_COL].values, "male": male})
        if v != AGE_COL:
            base["age_sd"] = age_sd.values
        d0 = pd.concat([base, X.reset_index(drop=True)], axis=1)
        Xi = X.reset_index(drop=True).mul(male, axis=0).add_suffix("__x_male")
        d1 = pd.concat([d0, Xi], axis=1)
        try:
            m0 = CoxPHFitter().fit(d0, "T", "E")
            m1 = CoxPHFitter().fit(d1, "T", "E")
        except Exception as e:
            print(f"  {v}: not estimable ({type(e).__name__})")
            continue
        lr = 2 * (m1.log_likelihood_ - m0.log_likelihood_)
        p_int = float(stats.chi2.sf(lr, Xi.shape[1]))
        for col in X.columns:
            b_x, b_i = m1.params_[col], m1.params_[f"{col}__x_male"]
            V = m1.variance_matrix_
            se_m = np.sqrt(V.loc[col, col] + V.loc[f"{col}__x_male", f"{col}__x_male"] + 2 * V.loc[col, f"{col}__x_male"])
            int_rows.append({"variable": v, "term": col, "label": LABELS.get(col, col),
                             "scale": "per SD" if (X.shape[1] == 1 and v not in multi_cols and full[cols_[0]].nunique() > 2) else "indicator",
                             "HR_women": np.exp(b_x), "HR_women_lo": np.exp(b_x - 1.96 * m1.standard_errors_[col]),
                             "HR_women_hi": np.exp(b_x + 1.96 * m1.standard_errors_[col]),
                             "HR_men": np.exp(b_x + b_i), "HR_men_lo": np.exp(b_x + b_i - 1.96 * se_m),
                             "HR_men_hi": np.exp(b_x + b_i + 1.96 * se_m),
                             "ratio_of_HR_men_vs_women": np.exp(b_i), "LR_chi2": lr, "df": Xi.shape[1], "p_interaction": p_int})
    sexint = pd.DataFrame(int_rows)
    if len(sexint):
        pv = sexint.drop_duplicates("variable").set_index("variable")["p_interaction"]
        order_ = pv.sort_values()
        q = (order_ * len(order_) / np.arange(1, len(order_) + 1))[::-1].cummin()[::-1].clip(upper=1)   # Benjamini-Hochberg
        sexint["q_BH"] = sexint["variable"].map(q)
    save_table(sexint, "eTable17_sex_interaction_tests", suppress=False)
    display(sexint.round(3))
    tick("Cox HR and interactions")

## Section 23. Data checks: CVD variables (R2.4); SF-36 definition (R2.3)

Both reviewers asked about definitions that can only be confirmed from the data in SURE. Only aggregate counts
are written; counts below SMALL_CELL are shown as "<5". Section 23.2 is optional if the data manager has already
documented the SF-36 scoring.

In [ ]:
# [Cell 84] [New] Settings for section 23
# Purpose: R2.3, R2.4. The raw file holds only the variables in 'Valid Data' (no CVD subtype, CVD survey or
#          diagnosis-date columns), so 23.1 reports the CVD-related predictors and lists any date-like columns.
CVD_COLS = ["diagnosisP", "heartdis", "AF"]
PF_ITEMS = ["limitvigor", "limitmod", "limitlift", "limitupfew", "limitupone",
            "limitbend", "limit1km", "limithalfkm", "limit100m", "limitdress"]   # data dictionary: "SF-36 (part)"
PF_CODES = {1: 0.0, 2: 50.0, 3: 100.0}   # 1 = limited a lot, 2 = limited a little, 3 = not limited (FACTIV.)

def small(n):
    n = int(n)
    return f"<{SMALL_CELL}" if 0 < n < SMALL_CELL else n

RAW = pd.read_csv(RAW_FILE, low_memory=False) if (RUN["data_checks"] and RAW_FILE.exists()) else None
print("raw file:", RAW_FILE, "->", "not used" if RAW is None else RAW.shape)

In [ ]:
# [Cell 85] [New] 23.1 CVD variables: is subtype information available; is the primary diagnosis code a model predictor;
#       which date / year columns exist (time from CVD diagnosis to enrolment)
# Purpose: R2.4.
if RUN["data_checks"]:
    print("in the analysis file?", {v: v in DF["bothsex"].columns for v in ["diagnosisP", "heartdis", "AF"]})
    if RAW is not None:
        rows = []
        for col in CVD_COLS:
            if col not in RAW.columns:
                rows.append({"variable": col, "value": "(not in file)", "n": ""})
                continue
            vc = RAW[col].astype(str).value_counts(dropna=False)
            for v, n in vc.head(30).items():
                rows.append({"variable": col, "value": v, "n": small(n)})
            if len(vc) > 30:
                rows.append({"variable": col, "value": f"other ({len(vc) - 30} values)", "n": small(vc.iloc[30:].sum())})
        check_cvd = pd.DataFrame(rows)
        check_cvd.to_csv(TABLES / "check_cvd_variables.csv", index=False)
        display(check_cvd)
        date_like = [c for c in RAW.columns if any(k in c.lower() for k in ["date", "year", "diag", "cvd", "admi", "episode", "stay"])]
        print("columns that might give the date of first CVD diagnosis:", date_like)
        pd.DataFrame({"column": date_like, "non_missing_n": [small(RAW[c].notna().sum()) for c in date_like]}).to_csv(
            TABLES / "check_date_columns.csv", index=False)

In [ ]:
# [Cell 86] [New] 23.2 (optional) Is sf36score the PF-10 physical functioning scale; how were incomplete items handled
# Purpose: R2.3 (scale, items, scoring, range, missing items).
if RUN["data_checks"]:
    df = DF["bothsex"]
    items = [c for c in PF_ITEMS if c in df.columns]
    print("PF items present:", len(items), "of 10")
    if len(items) == 10 and "sf36score" in df.columns:
        it = df[items].apply(pd.to_numeric, errors="coerce").where(lambda x: x.isin(list(PF_CODES)))
        pts = it.replace(PF_CODES)
        answered = it.notna().sum(axis=1)
        sf = pd.to_numeric(df["sf36score"], errors="coerce").where(lambda x: x.between(0, 100))
        pf_all = pts.mean(axis=1).where(answered == 10)
        pf_half = pts.mean(axis=1).where(answered >= 5)          # usual SF-36 rule: at least half the items answered
        out = [("n participants", small(len(df))), ("n sf36score valid (0-100)", small(sf.notna().sum())),
               ("n all 10 items answered", small((answered == 10).sum())),
               ("n 5-9 items answered", small(answered.between(5, 9).sum())), ("n <5 items answered", small((answered < 5).sum()))]
        for lab, pf, mask in [("all 10 answered", pf_all, answered == 10), ("5-9 answered (half rule)", pf_half, answered.between(5, 9))]:
            m = mask & sf.notna() & pf.notna()
            out += [(f"{lab}: n compared", small(m.sum())),
                    (f"{lab}: correlation with sf36score", round(float(np.corrcoef(pf[m], sf[m])[0, 1]), 4) if m.sum() > 2 else np.nan),
                    (f"{lab}: share |PF - sf36score| <= 0.5", round(float((np.abs(pf[m] - sf[m]) <= 0.5).mean()), 4) if m.any() else np.nan),
                    (f"{lab}: mean |PF - sf36score|", round(float(np.abs(pf[m] - sf[m]).mean()), 3) if m.any() else np.nan)]
        for k in range(0, 11):
            mk = answered == k
            if mk.any():
                out.append((f"items answered = {k}: n / sf36score missing", f"{small(mk.sum())} / {small((mk & sf.isna()).sum())}"))
        check_sf36 = pd.DataFrame(out, columns=["check", "value"])
        check_sf36.to_csv(TABLES / "check_sf36_definition.csv", index=False)
        display(check_sf36)
        print("Reading: correlation near 1 and share |diff| <= 0.5 near 1 -> sf36score is the PF-10 scale (0-100, higher = "
              "better physical functioning); correlation near -1 -> sf36score is reverse-coded (higher = worse functioning). "
              "If people answering 5-9 items still have a score that matches the half rule, "
              "the missing-item rule is 'at least half of the items answered'.")
    tick("data checks")

## Section 24. Numbers for the text and export list

In [ ]:
# [Cell 87] [New] Key numbers, run settings and the list of files to export
# Purpose: fills the placeholders in the response letter; documents the run.
key = {"versions": VERSIONS_USED, "quick_test": QUICK_TEST, "cohorts": COHORTS_TO_RUN,
       "settings": {"N_BOOT": N_BOOT, "N_BOOT_OPT": N_BOOT_OPT, "N_BOOT_SECONDARY": N_BOOT_SECONDARY,
                    "CV": f"{CV_FOLDS}x{CV_REPEATS}", "nested_CV": f"{NESTED_FOLDS}x{NESTED_REPEATS}",
                    "M_IMPUTATIONS": M_IMPUTATIONS, "N_SPLIT_SEEDS": N_SPLIT_SEEDS, "N_SHAP_BOOT": N_SHAP_BOOT,
                    "PRIMARY_SCHEME": PRIMARY_SCHEME, "RUN": RUN},
       "timings_min": TIMINGS}
for c in COHORTS_TO_RUN:
    key[c] = {}
    for sch in SCHEMES:
        t1 = calib[(calib.cohort == c) & (calib.horizon == PRIMARY) & (calib.scheme == sch)].set_index("version")
        key[c][sch] = {v: {m: t1.loc[v, f"{m}_fmt"] for m in ["AUROC", "PR_AUC", "Brier", "cal_intercept", "cal_slope"]}
                       for v in ["original", "Platt", "isotonic"]}
        key[c][sch]["paired_auroc_diff_10y"] = {r.comparison: r.fmt for r in diffs[(diffs.cohort == c) & (diffs.horizon == PRIMARY)
                                                                                    & (diffs.scheme == sch)].itertuples()}
        key[c][sch]["dca_useful_range_10y"] = {k[3]: v for k, v in DCA_RANGE.items() if k[0] == c and k[1] == PRIMARY and k[2] == sch
                                               and k[3].startswith("LightGBM")}
(TABLES / "key_numbers_for_text.json").write_text(json.dumps(key, indent=2, default=str))
print(json.dumps({k: v for k, v in key.items() if k in COHORTS_TO_RUN}, indent=1, default=str)[:4000])
print("\nExport (aggregate only):")
for p in sorted(list(TABLES.glob("*")) + list(FIGS.glob("*"))):
    print("  ", p.relative_to(OUT_DIR))
print("Do NOT export:", ROWLEVEL)
tick("finished")